# Leadership and Management Book Recommendation System

## Notebook 03 — Data Cleaning

### Purpose

This notebook cleans and validates the raw book datasets collected from the two primary data sources:

1. **Open Library API**
2. **LeadershipNow / LeaderShop Web Scraping**

The objective is to transform the raw collected data into consistent, validated, and analysis-ready datasets while preserving the original source information.

The two sources will be cleaned independently in this notebook.

They will **not** be merged at this stage. Cross-source matching, deduplication, and integration will be performed in Notebook 04.

---

### Cleaning Objectives

This notebook will:

- load and inspect all relevant raw datasets;
- preserve the raw source files without modification;
- standardize missing-value representations;
- inspect and clean text fields;
- validate ISBN identifiers;
- investigate malformed or unusual ISBN values;
- standardize publication dates;
- derive publication year where appropriate;
- parse book format and page-count information;
- investigate duplicate records;
- validate numerical fields;
- inspect categorical fields;
- document all cleaning decisions;
- generate cleaned source-specific datasets.

---

### Data Cleaning Principle

Raw source files will never be overwritten.

All transformations will be performed on copies of the raw datasets.

Missing or questionable information will not be fabricated.

Where a value cannot be reliably corrected from the available source data, it will remain missing or will be flagged for later investigation.

## 2. Data Cleaning Workflow

The cleaning process follows the sequence below:

### Open Library

Raw Search API data  
→ schema validation  
→ missing-value assessment  
→ text cleaning  
→ ISBN assessment  
→ date/year validation  
→ numerical-field validation  
→ duplicate investigation  
→ cleaned Open Library dataset

### Open Library Work Enrichment

Raw Work API enrichment  
→ schema validation  
→ text-field assessment  
→ subject-field assessment  
→ missing-value validation  
→ cleaned enrichment dataset

### LeadershipNow

Raw scraped data  
→ schema validation  
→ missing-value assessment  
→ text cleaning  
→ ISBN validation  
→ publication-date parsing  
→ format parsing  
→ page-count extraction  
→ duplicate investigation  
→ cleaned LeadershipNow dataset

### Output

The cleaned source-specific datasets will be saved in:

`data/processed/`

Cross-source integration will be performed separately in Notebook 04.

In [1]:
import pathlib
import re

import numpy as np
import pandas as pd

In [2]:
print(
    "Libraries imported successfully."
)

Libraries imported successfully.


In [3]:
project_path = pathlib.Path.cwd().parent

raw_data_path = (
    project_path
    / "data"
    / "raw"
)

processed_data_path = (
    project_path
    / "data"
    / "processed"
)

processed_data_path.mkdir(
    parents=True,
    exist_ok=True
)

print(
    "Project path:",
    project_path
)

print(
    "Raw data path:",
    raw_data_path
)

print(
    "Processed data path:",
    processed_data_path
)

print(
    "Raw folder exists:",
    raw_data_path.exists()
)

print(
    "Processed folder exists:",
    processed_data_path.exists()
)

Project path: /Users/jannoelvero/Documents/Ironhack/Week_11/Leadership_Management_Book_Recommendation_System
Raw data path: /Users/jannoelvero/Documents/Ironhack/Week_11/Leadership_Management_Book_Recommendation_System/data/raw
Processed data path: /Users/jannoelvero/Documents/Ironhack/Week_11/Leadership_Management_Book_Recommendation_System/data/processed
Raw folder exists: True
Processed folder exists: True


In [4]:
raw_files = sorted(
    raw_data_path.glob("*.csv")
)

print(
    "CSV files found:",
    len(raw_files)
)

for file in raw_files:
    print(
        "-",
        file.name
    )

CSV files found: 8
- bookshop_collection_evaluation.csv
- leadershipnow_books_raw.csv
- leadershipnow_cover_anomalies.csv
- leadershipnow_scraping_log.csv
- open_library_collection_log.csv
- open_library_search_api_raw.csv
- open_library_work_enrichment_checkpoint.csv
- open_library_work_enrichment_raw.csv


In [5]:
openlibrary_search_raw = pd.read_csv(
    raw_data_path
    / "open_library_search_api_raw.csv"
)

openlibrary_work_raw = pd.read_csv(
    raw_data_path
    / "open_library_work_enrichment_raw.csv"
)

leadershipnow_raw = pd.read_csv(
    raw_data_path
    / "leadershipnow_books_raw.csv"
)

In [6]:
print(
    "Open Library Search:",
    openlibrary_search_raw.shape
)

print(
    "Open Library Work Enrichment:",
    openlibrary_work_raw.shape
)

print(
    "LeadershipNow:",
    leadershipnow_raw.shape
)

Open Library Search: (1000, 31)
Open Library Work Enrichment: (950, 13)
LeadershipNow: (1124, 15)


In [7]:
openlibrary_search_clean = (
    openlibrary_search_raw.copy()
)

openlibrary_work_clean = (
    openlibrary_work_raw.copy()
)

leadershipnow_clean = (
    leadershipnow_raw.copy()
)

In [8]:
print(
    "Raw datasets copied successfully."
)

print(
    openlibrary_search_clean.shape
)

print(
    openlibrary_work_clean.shape
)

print(
    leadershipnow_clean.shape
)

Raw datasets copied successfully.
(1000, 31)
(950, 13)
(1124, 15)


### Raw-Data Protection

The raw datasets were loaded into memory and immediately copied before any transformations were applied.

All subsequent cleaning operations will be performed on variables ending in `_clean`.

The original `_raw` DataFrames remain unchanged and can therefore be used to verify transformations throughout the notebook.

In [9]:
dataset_inventory = pd.DataFrame({
    "dataset": [
        "Open Library Search API",
        "Open Library Work Enrichment",
        "LeadershipNow"
    ],

    "rows": [
        len(openlibrary_search_raw),
        len(openlibrary_work_raw),
        len(leadershipnow_raw)
    ],

    "columns": [
        openlibrary_search_raw.shape[1],
        openlibrary_work_raw.shape[1],
        leadershipnow_raw.shape[1]
    ],

    "duplicate_rows": [
        openlibrary_search_raw.duplicated().sum(),
        openlibrary_work_raw.duplicated().sum(),
        leadershipnow_raw.duplicated().sum()
    ]
})

dataset_inventory

,dataset,rows,columns,duplicate_rows
0,Open Library Search API,1000,31,0
1,Open Library Work Enrichment,950,13,0
2,LeadershipNow,1124,15,0


In [10]:
print(
    "OPEN LIBRARY SEARCH COLUMNS"
)

print(
    openlibrary_search_raw.columns.tolist()
)

OPEN LIBRARY SEARCH COLUMNS
['openlibrary_key', 'title', 'authors', 'author_keys', 'first_publish_year', 'publish_dates', 'publishers', 'isbn_10', 'isbn_13', 'all_isbns', 'languages', 'subjects', 'edition_count', 'ratings_average', 'ratings_count', 'ratings_count_1', 'ratings_count_2', 'ratings_count_3', 'ratings_count_4', 'ratings_count_5', 'want_to_read_count', 'currently_reading_count', 'already_read_count', 'cover_id', 'cover_url', 'ebook_access', 'has_fulltext', 'public_scan', 'collection_query', 'source', 'source_url']


In [11]:
print(
    "OPEN LIBRARY WORK ENRICHMENT COLUMNS"
)

print(
    openlibrary_work_raw.columns.tolist()
)

OPEN LIBRARY WORK ENRICHMENT COLUMNS
['openlibrary_key', 'title', 'work_status_code', 'description', 'first_sentence', 'work_subjects', 'subject_places', 'subject_people', 'subject_times', 'excerpts', 'lc_classifications', 'dewey_number', 'work_covers']


In [12]:
print(
    "LEADERSHIPNOW COLUMNS"
)

print(
    leadershipnow_raw.columns.tolist()
)

LEADERSHIPNOW COLUMNS
['scrape_id', 'title', 'subtitle', 'author', 'format_raw', 'isbn', 'publisher', 'publication_date_raw', 'cover_url', 'cover_alt_raw', 'external_book_url', 'release_page', 'release_page_url', 'source', 'scraped_at']


In [13]:
print(
    "OPEN LIBRARY SEARCH"
)

openlibrary_search_raw.info()

OPEN LIBRARY SEARCH
<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 31 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   openlibrary_key          1000 non-null   str    
 1   title                    1000 non-null   str    
 2   authors                  992 non-null    str    
 3   author_keys              992 non-null    str    
 4   first_publish_year       997 non-null    float64
 5   publish_dates            997 non-null    str    
 6   publishers               994 non-null    str    
 7   isbn_10                  1000 non-null   str    
 8   isbn_13                  1000 non-null   str    
 9   all_isbns                1000 non-null   str    
 10  languages                952 non-null    str    
 11  subjects                 875 non-null    str    
 12  edition_count            1000 non-null   int64  
 13  ratings_average          323 non-null    float64
 14  ratings_count   

In [14]:
print(
    "OPEN LIBRARY WORK ENRICHMENT"
)

openlibrary_work_raw.info()

OPEN LIBRARY WORK ENRICHMENT
<class 'pandas.DataFrame'>
RangeIndex: 950 entries, 0 to 949
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   openlibrary_key     950 non-null    str  
 1   title               950 non-null    str  
 2   work_status_code    950 non-null    int64
 3   description         192 non-null    str  
 4   first_sentence      32 non-null     str  
 5   work_subjects       826 non-null    str  
 6   subject_places      56 non-null     str  
 7   subject_people      13 non-null     str  
 8   subject_times       9 non-null      str  
 9   excerpts            950 non-null    str  
 10  lc_classifications  85 non-null     str  
 11  dewey_number        155 non-null    str  
 12  work_covers         743 non-null    str  
dtypes: int64(1), str(12)
memory usage: 96.6 KB


In [15]:
print(
    "LEADERSHIPNOW"
)

leadershipnow_raw.info()

LEADERSHIPNOW
<class 'pandas.DataFrame'>
RangeIndex: 1124 entries, 0 to 1123
Data columns (total 15 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   scrape_id             1124 non-null   int64
 1   title                 1124 non-null   str  
 2   subtitle              1081 non-null   str  
 3   author                1124 non-null   str  
 4   format_raw            1124 non-null   str  
 5   isbn                  1124 non-null   str  
 6   publisher             1124 non-null   str  
 7   publication_date_raw  1124 non-null   str  
 8   cover_url             1124 non-null   str  
 9   cover_alt_raw         1124 non-null   str  
 10  external_book_url     1124 non-null   str  
 11  release_page          1124 non-null   str  
 12  release_page_url      1124 non-null   str  
 13  source                1124 non-null   str  
 14  scraped_at            1124 non-null   str  
dtypes: int64(1), str(14)
memory usage: 131.8 KB


In [16]:
def missing_value_summary(df):

    summary = pd.DataFrame({
        "data_type": df.dtypes.astype(str),

        "missing_count":
            df.isna().sum(),

        "missing_pct":
            (
                df.isna().mean()
                * 100
            ).round(2),

        "unique_values":
            df.nunique(
                dropna=True
            )
    })

    return summary.sort_values(
        "missing_pct",
        ascending=False
    )

In [17]:
openlibrary_search_missing = (
    missing_value_summary(
        openlibrary_search_clean
    )
)

openlibrary_search_missing

,data_type,missing_count,missing_pct,unique_values
ratings_count_1,float64,677,67.7,7
ratings_count_5,float64,677,67.7,16
ratings_count_4,float64,677,67.7,11
ratings_count_3,float64,677,67.7,10
ratings_count_2,float64,677,67.7,4
ratings_count,float64,677,67.7,24
ratings_average,float64,677,67.7,51
cover_url,str,182,18.2,771
cover_id,float64,182,18.2,771
want_to_read_count,float64,143,14.3,134


In [18]:
openlibrary_work_missing = (
    missing_value_summary(
        openlibrary_work_clean
    )
)

openlibrary_work_missing

,data_type,missing_count,missing_pct,unique_values
subject_times,str,941,99.05,9
subject_people,str,937,98.63,13
first_sentence,str,918,96.63,31
subject_places,str,894,94.11,18
lc_classifications,str,865,91.05,85
dewey_number,str,795,83.68,76
description,str,758,79.79,192
work_covers,str,207,21.79,742
work_subjects,str,124,13.05,725
openlibrary_key,str,0,0.00,950


In [19]:
leadershipnow_missing = (
    missing_value_summary(
        leadershipnow_clean
    )
)

leadershipnow_missing

,data_type,missing_count,missing_pct,unique_values
subtitle,str,43,3.83,1075
scrape_id,int64,0,0.00,1124
title,str,0,0.00,1114
author,str,0,0.00,1087
format_raw,str,0,0.00,196
isbn,str,0,0.00,1120
publisher,str,0,0.00,302
publication_date_raw,str,0,0.00,343
cover_url,str,0,0.00,1120
cover_alt_raw,str,0,0.00,1120


## Initial Cleaning Audit

The initial audit examines dataset dimensions, inferred data types, exact duplicate rows, missing values, and unique-value counts before transformations are applied.

Missing data will be evaluated according to the meaning of each field rather than using a single blanket treatment.

For example:

- missing descriptions will not automatically be replaced with synthetic text;
- missing ratings will not automatically be interpreted as zero ratings;
- missing subtitles may legitimately indicate that a book has no subtitle;
- missing subject metadata will remain missing unless another collected source provides verifiable information;
- identifier anomalies will be investigated before correction or removal.

The purpose of this stage is therefore to distinguish between:

1. legitimate absence of information;
2. structural missingness;
3. malformed values;
4. duplicated source records;
5. fields requiring normalization.

No rows are removed during this initial audit.

## 13. Open Library Search API — Cleaning Strategy

The Open Library Search dataset contains 1,000 collected records representing 950 unique Open Library works.

The collection intentionally retained records returned by multiple search queries. Therefore, repeated `openlibrary_key` values are not treated as accidental raw-data corruption.

Several Open Library API fields originally contained lists. After export to CSV, these fields were stored as string representations of Python lists.

Examples include:

- authors
- author keys
- publication dates
- publishers
- ISBN values
- languages
- subjects

These fields must be reconstructed carefully before further processing.

The cleaning sequence will therefore:

1. inspect serialized list structures;
2. safely reconstruct list values;
3. preserve missing information;
4. normalize basic text;
5. validate publication-year fields;
6. validate numerical metrics;
7. investigate repeated Open Library works;
8. select representative records only when methodologically justified.

In [20]:
list_like_columns = [
    "authors",
    "author_keys",
    "publish_dates",
    "publishers",
    "isbn_10",
    "isbn_13",
    "all_isbns",
    "languages",
    "subjects"
]

for column in list_like_columns:

    print(
        f"\n--- {column} ---"
    )

    print(
        openlibrary_search_clean[
            column
        ].dropna().head(3).tolist()
    )


--- authors ---
["['Stephen R. Covey']", "['Gary A. Yukl']", "['Karjadi M.']"]

--- author_keys ---
["['OL383159A']", "['OL400156A']", "['OL1268A']"]

--- publish_dates ---
["['January 1, 1992', 'October 1, 2001', 'March 1, 1992', '1991', 'September 1991', 'January 1, 2002', 'April 1989', 'January 4, 1999', 'March 1, 2005', 'October 1, 1992', '2002', '2003', 'March 7, 2005', 'January 1, 2000', 'Apr 01, 2012', '1992', 'October 26, 1992']", "['2010', '2007', '1994', '2013', '1989', '2017-01-01', 'June 7, 2001', '2010 7th Ed', '2002', 'June 2001', '2009', '2005', '2012', '2003', '2020', '1981', '1998', 'Oct 28, 2018', 'Jan 14, 2019']", "['1977']"]

--- publishers ---
["['Franklin Covey on Brilliance Audio', 'Free Press', 'Covey', 'Coach Series', 'Pocket Books', 'Simon & Schuster Audio', 'Simon Schuster Trade', 'Cedar Fort', 'Simon & Schuster Inc.', 'Covey Leadership Center', 'Simon & Schuster (Trade Division)', 'Simon & Schuster', 'Summit Books']", "['Prentice Hall', 'Prentice-Hall Inter

In [21]:
for column in list_like_columns:

    sample = (
        openlibrary_search_clean[
            column
        ]
        .dropna()
        .iloc[0]
    )

    print(
        column,
        "| Python type:",
        type(sample).__name__,
        "| Value:",
        sample
    )

authors | Python type: str | Value: ['Stephen R. Covey']
author_keys | Python type: str | Value: ['OL383159A']
publish_dates | Python type: str | Value: ['January 1, 1992', 'October 1, 2001', 'March 1, 1992', '1991', 'September 1991', 'January 1, 2002', 'April 1989', 'January 4, 1999', 'March 1, 2005', 'October 1, 1992', '2002', '2003', 'March 7, 2005', 'January 1, 2000', 'Apr 01, 2012', '1992', 'October 26, 1992']
publishers | Python type: str | Value: ['Franklin Covey on Brilliance Audio', 'Free Press', 'Covey', 'Coach Series', 'Pocket Books', 'Simon & Schuster Audio', 'Simon Schuster Trade', 'Cedar Fort', 'Simon & Schuster Inc.', 'Covey Leadership Center', 'Simon & Schuster (Trade Division)', 'Simon & Schuster', 'Summit Books']
isbn_10 | Python type: str | Value: ['002863912X', '0671011138', '068485841X', '0743501551', '1883219248', '1929494610', '0916095312', '1596590084', '0671792806', '155517048X', '0671749102', '0671317032', '145589348X', '188321906X', '0671755455', '0671711350'

In [22]:
import ast

In [23]:
def parse_serialized_list(value):
    """
    Convert a string representation of a Python list
    back into a list.

    Missing values remain missing.
    Existing lists are returned unchanged.
    Values that cannot be safely parsed are preserved
    for later inspection.
    """

    if isinstance(value, list):
        return value

    if pd.isna(value):
        return np.nan

    if isinstance(value, str):

        value = value.strip()

        try:
            parsed = ast.literal_eval(
                value
            )

            if isinstance(parsed, list):
                return parsed

        except (
            ValueError,
            SyntaxError
        ):
            pass

    return value

In [24]:
for column in list_like_columns:

    sample = (
        openlibrary_search_clean[
            column
        ]
        .dropna()
        .iloc[0]
    )

    parsed_sample = (
        parse_serialized_list(
            sample
        )
    )

    print(
        f"\n{column}"
    )

    print(
        "Before:",
        sample
    )

    print(
        "After:",
        parsed_sample
    )

    print(
        "Type:",
        type(
            parsed_sample
        ).__name__
    )


authors
Before: ['Stephen R. Covey']
After: ['Stephen R. Covey']
Type: list

author_keys
Before: ['OL383159A']
After: ['OL383159A']
Type: list

publish_dates
Before: ['January 1, 1992', 'October 1, 2001', 'March 1, 1992', '1991', 'September 1991', 'January 1, 2002', 'April 1989', 'January 4, 1999', 'March 1, 2005', 'October 1, 1992', '2002', '2003', 'March 7, 2005', 'January 1, 2000', 'Apr 01, 2012', '1992', 'October 26, 1992']
After: ['January 1, 1992', 'October 1, 2001', 'March 1, 1992', '1991', 'September 1991', 'January 1, 2002', 'April 1989', 'January 4, 1999', 'March 1, 2005', 'October 1, 1992', '2002', '2003', 'March 7, 2005', 'January 1, 2000', 'Apr 01, 2012', '1992', 'October 26, 1992']
Type: list

publishers
Before: ['Franklin Covey on Brilliance Audio', 'Free Press', 'Covey', 'Coach Series', 'Pocket Books', 'Simon & Schuster Audio', 'Simon Schuster Trade', 'Cedar Fort', 'Simon & Schuster Inc.', 'Covey Leadership Center', 'Simon & Schuster (Trade Division)', 'Simon & Schuste

In [25]:
for column in list_like_columns:

    openlibrary_search_clean[
        column
    ] = (
        openlibrary_search_clean[
            column
        ].apply(
            parse_serialized_list
        )
    )

In [26]:
for column in list_like_columns:

    non_missing = (
        openlibrary_search_clean[
            column
        ].dropna()
    )

    if len(non_missing) > 0:

        print(
            column,
            "->",
            type(
                non_missing.iloc[0]
            ).__name__
        )

authors -> list
author_keys -> list
publish_dates -> list
publishers -> list
isbn_10 -> list
isbn_13 -> list
all_isbns -> list
languages -> list
subjects -> list


In [27]:
list_parsing_audit = []

for column in list_like_columns:

    non_missing = (
        openlibrary_search_clean[
            column
        ].dropna()
    )

    list_count = (
        non_missing
        .apply(
            lambda value:
                isinstance(
                    value,
                    list
                )
        )
        .sum()
    )

    non_list_count = (
        len(non_missing)
        - list_count
    )

    list_parsing_audit.append({
        "column": column,
        "non_missing_values":
            len(non_missing),
        "parsed_as_list":
            list_count,
        "not_parsed_as_list":
            non_list_count
    })

list_parsing_audit_df = (
    pd.DataFrame(
        list_parsing_audit
    )
)

list_parsing_audit_df

,column,non_missing_values,parsed_as_list,not_parsed_as_list
0,authors,992,992,0
1,author_keys,992,992,0
2,publish_dates,997,997,0
3,publishers,994,994,0
4,isbn_10,1000,1000,0
5,isbn_13,1000,1000,0
6,all_isbns,1000,1000,0
7,languages,952,952,0
8,subjects,875,875,0


In [28]:
isbn_availability = pd.DataFrame({
    "field": [
        "isbn_10",
        "isbn_13",
        "all_isbns"
    ],

    "records_with_values": [
        openlibrary_search_clean[
            "isbn_10"
        ].apply(
            lambda x:
                isinstance(x, list)
                and len(x) > 0
        ).sum(),

        openlibrary_search_clean[
            "isbn_13"
        ].apply(
            lambda x:
                isinstance(x, list)
                and len(x) > 0
        ).sum(),

        openlibrary_search_clean[
            "all_isbns"
        ].apply(
            lambda x:
                isinstance(x, list)
                and len(x) > 0
        ).sum()
    ]
})

isbn_availability[
    "records_without_values"
] = (
    len(openlibrary_search_clean)
    -
    isbn_availability[
        "records_with_values"
    ]
)

isbn_availability

,field,records_with_values,records_without_values
0,isbn_10,951,49
1,isbn_13,964,36
2,all_isbns,964,36


In [29]:
work_frequency = (
    openlibrary_search_clean[
        "openlibrary_key"
    ]
    .value_counts()
)

print(
    "Total rows:",
    len(openlibrary_search_clean)
)

print(
    "Unique works:",
    openlibrary_search_clean[
        "openlibrary_key"
    ].nunique()
)

print(
    "Repeated work keys:",
    (
        work_frequency > 1
    ).sum()
)

print(
    "Maximum appearances of one work:",
    work_frequency.max()
)

Total rows: 1000
Unique works: 950
Repeated work keys: 42
Maximum appearances of one work: 6


In [30]:
work_frequency.value_counts().sort_index()

count
1    908
2     37
3      4
6      1
Name: count, dtype: int64

In [31]:
repeated_keys = (
    work_frequency[
        work_frequency > 1
    ].index
)

repeated_works_df = (
    openlibrary_search_clean[
        openlibrary_search_clean[
            "openlibrary_key"
        ].isin(
            repeated_keys
        )
    ]
    .sort_values(
        [
            "openlibrary_key",
            "collection_query"
        ]
    )
)

print(
    "Rows belonging to repeated works:",
    len(repeated_works_df)
)

repeated_works_df[
    [
        "openlibrary_key",
        "title",
        "authors",
        "collection_query",
        "ratings_average",
        "ratings_count"
    ]
].head(30)

Rows belonging to repeated works: 92


,openlibrary_key,title,authors,collection_query,ratings_average,ratings_count
328,/works/OL102817W,A Guide to the Project Management Body of Know...,[Project Management Institute],management,5.000000,2.0
551,/works/OL102817W,A Guide to the Project Management Body of Know...,[Project Management Institute],project management,5.000000,2.0
692,/works/OL11274839W,Human resource management at work,"[Mick Marchington, Mick Marchington, Adrian Wi...",human resource management,NaN,NaN
355,/works/OL11274839W,Human resource management at work,"[Mick Marchington, Mick Marchington, Adrian Wi...",people management,NaN,NaN
650,/works/OL11789290W,Human Resource management,[Gary Dessler],human resource management,4.125000,8.0
303,/works/OL11789290W,Human Resource management,[Gary Dessler],management,4.125000,8.0
982,/works/OL16002869W,Strategic management,"[Gregory G. Dess, G. T. Lumpkin, Alan B. Eisne...",innovation management,4.200000,5.0
309,/works/OL16002869W,Strategic management,"[Gregory G. Dess, G. T. Lumpkin, Alan B. Eisne...",management,4.200000,5.0
401,/works/OL16002869W,Strategic management,"[Gregory G. Dess, G. T. Lumpkin, Alan B. Eisne...",strategic management,4.200000,5.0
140,/works/OL17934890W,Leadership In Turbulent Times,"[Doris Kearns Goodwin, Doris Kearns Goodwin]",executive leadership,NaN,NaN


In [32]:
consistency_columns = [
    "title",
    "first_publish_year",
    "edition_count",
    "ratings_average",
    "ratings_count",
    "cover_id"
]

repeated_consistency = []

for key, group in (
    repeated_works_df.groupby(
        "openlibrary_key"
    )
):

    result = {
        "openlibrary_key": key,
        "appearances": len(group)
    }

    for column in consistency_columns:

        result[
            f"{column}_unique"
        ] = (
            group[column]
            .nunique(
                dropna=False
            )
        )

    repeated_consistency.append(
        result
    )

repeated_consistency_df = (
    pd.DataFrame(
        repeated_consistency
    )
)

repeated_consistency_df.head()

,openlibrary_key,appearances,title_unique,first_publish_year_unique,edition_count_unique,ratings_average_unique,ratings_count_unique,cover_id_unique
0,/works/OL102817W,2,1,1,1,1,1,1
1,/works/OL11274839W,2,1,1,1,1,1,1
2,/works/OL11789290W,2,1,1,1,1,1,1
3,/works/OL16002869W,3,1,1,1,1,1,1
4,/works/OL17934890W,2,1,1,1,1,1,1


In [33]:
for column in consistency_columns:

    inconsistent_count = (
        repeated_consistency_df[
            f"{column}_unique"
        ] > 1
    ).sum()

    print(
        column,
        "| inconsistent repeated works:",
        inconsistent_count
    )

title | inconsistent repeated works: 0
first_publish_year | inconsistent repeated works: 0
edition_count | inconsistent repeated works: 0
ratings_average | inconsistent repeated works: 0
ratings_count | inconsistent repeated works: 0
cover_id | inconsistent repeated works: 0


## 23. Serialized List Reconstruction

Several fields returned by the Open Library API originally contained arrays but were stored as string representations of lists after CSV export.

The following fields were successfully reconstructed:

- authors
- author_keys
- publish_dates
- publishers
- isbn_10
- isbn_13
- all_isbns
- languages
- subjects

All non-missing values in these fields were successfully parsed as Python lists.

No parsing failures were identified.

### ISBN Availability

After reconstructing the list fields:

- 951 of 1,000 collected rows contain at least one ISBN-10;
- 964 contain at least one ISBN-13;
- 964 contain at least one ISBN of either type;
- 36 collected rows contain no ISBN values.

Empty ISBN lists are treated differently from missing CSV values. An empty list indicates that the API returned no ISBN for the work.

In [34]:
query_membership = (
    openlibrary_search_clean
    .groupby(
        "openlibrary_key"
    )["collection_query"]
    .agg(
        lambda values:
            sorted(
                set(values)
            )
    )
    .rename(
        "collection_queries"
    )
)

query_membership.head()

openlibrary_key
/works/OL102817W                    [management, project management]
/works/OL10932218W                            [executive leadership]
/works/OL11019317W                          [leadership development]
/works/OL11150616W                            [executive leadership]
/works/OL11274839W    [human resource management, people management]
Name: collection_queries, dtype: object

In [35]:
query_match_count = (
    query_membership
    .apply(len)
    .rename(
        "query_match_count"
    )
)

query_match_count.describe()

count    950.000000
mean       1.052632
std        0.281813
min        1.000000
25%        1.000000
50%        1.000000
75%        1.000000
max        6.000000
Name: query_match_count, dtype: float64

In [36]:
openlibrary_search_unique = (
    openlibrary_search_clean
    .drop_duplicates(
        subset="openlibrary_key",
        keep="first"
    )
    .copy()
)

In [37]:
openlibrary_search_unique = (
    openlibrary_search_unique
    .drop(
        columns=[
            "collection_query"
        ]
    )
)

In [38]:
openlibrary_search_unique = (
    openlibrary_search_unique
    .merge(
        query_membership,
        on="openlibrary_key",
        how="left"
    )
)

In [39]:
openlibrary_search_unique = (
    openlibrary_search_unique
    .merge(
        query_match_count,
        on="openlibrary_key",
        how="left"
    )
)

In [40]:
print(
    "Original collected rows:",
    len(openlibrary_search_clean)
)

print(
    "Unique work records:",
    len(openlibrary_search_unique)
)

print(
    "Unique Open Library keys:",
    openlibrary_search_unique[
        "openlibrary_key"
    ].nunique()
)

print(
    "Duplicate Open Library keys:",
    openlibrary_search_unique[
        "openlibrary_key"
    ].duplicated().sum()
)

Original collected rows: 1000
Unique work records: 950
Unique Open Library keys: 950
Duplicate Open Library keys: 0


In [41]:
openlibrary_search_unique[
    [
        "title",
        "authors",
        "collection_queries",
        "query_match_count"
    ]
].sort_values(
    "query_match_count",
    ascending=False
).head(15)

,title,authors,collection_queries,query_match_count
1,Leadership in Organizations,[Gary A. Yukl],"[executive leadership, leadership, leadership ...",6
293,Human resource management,"[R. Wayne Mondy, Robert M. Noe, Shane R. Preme...","[human resource management, management, perfor...",3
289,Strategic management,"[Gregory G. Dess, G. T. Lumpkin, Alan B. Eisne...","[innovation management, management, strategic ...",3
290,Management,[Stephen P. Robbins],"[change management, innovation management, man...",3
295,Strategic management and business policy,"[Thomas L. Wheelen, J. David Hunger, Tom Wheelen]","[innovation management, management, strategic ...",3
311,Strategic management,"[Pearce, John A., John A. Pearce, Richard B. R...","[management, strategic management]",2
310,Strategic market management,[David A. Aaker],"[management, strategic management]",2
473,Operations management,[Roger G. Schroeder],"[decision making, operations management]",2
36,Improving organizational effectiveness through...,"[Bernard M. Bass, Bruce J. Avolio]","[leadership, transformational leadership]",2
280,Marketing management,[Philip Kotler],"[change management, management]",2


In [42]:
def clean_text_value(value):
    """
    Normalize surrounding and repeated whitespace
    without changing capitalization or wording.
    """

    if pd.isna(value):
        return np.nan

    value = str(value).strip()

    value = re.sub(
        r"\s+",
        " ",
        value
    )

    return value if value else np.nan

In [43]:
scalar_text_columns = [
    "openlibrary_key",
    "title",
    "ebook_access",
    "source",
    "source_url"
]

for column in scalar_text_columns:

    openlibrary_search_unique[
        column
    ] = (
        openlibrary_search_unique[
            column
        ].apply(
            clean_text_value
        )
    )

In [44]:
def clean_string_list(value):
    """
    Normalize whitespace within list elements
    while preserving the list structure.
    """

    if not isinstance(value, list):
        return value

    cleaned_values = []

    for item in value:

        if pd.isna(item):
            continue

        cleaned_item = (
            re.sub(
                r"\s+",
                " ",
                str(item).strip()
            )
        )

        if cleaned_item:
            cleaned_values.append(
                cleaned_item
            )

    return cleaned_values

In [45]:
for column in list_like_columns:

    openlibrary_search_unique[
        column
    ] = (
        openlibrary_search_unique[
            column
        ].apply(
            clean_string_list
        )
    )

In [46]:
def remove_list_duplicates(value):

    if not isinstance(value, list):
        return value

    return list(
        dict.fromkeys(value)
    )

In [47]:
for column in list_like_columns:

    openlibrary_search_unique[
        column
    ] = (
        openlibrary_search_unique[
            column
        ].apply(
            remove_list_duplicates
        )
    )

In [48]:
openlibrary_search_unique[
    "collection_queries"
] = (
    openlibrary_search_unique[
        "collection_queries"
    ].apply(
        remove_list_duplicates
    )
)

In [49]:
openlibrary_search_unique[
    "query_match_count"
] = (
    openlibrary_search_unique[
        "collection_queries"
    ].apply(len)
)

In [50]:
openlibrary_search_unique[
    "first_publish_year"
].describe()

count     947.000000
mean     2000.982049
std        16.087127
min      1900.000000
25%      1993.500000
50%      2004.000000
75%      2012.000000
max      2025.000000
Name: first_publish_year, dtype: float64

In [51]:
print(
    "Earliest year:",
    openlibrary_search_unique[
        "first_publish_year"
    ].min()
)

print(
    "Latest year:",
    openlibrary_search_unique[
        "first_publish_year"
    ].max()
)

Earliest year: 1900.0
Latest year: 2025.0


In [52]:
openlibrary_search_unique.loc[
    (
        openlibrary_search_unique[
            "first_publish_year"
        ] < 1900
    )
    |
    (
        openlibrary_search_unique[
            "first_publish_year"
        ] > 2026
    ),
    [
        "title",
        "authors",
        "first_publish_year",
        "publish_dates",
        "openlibrary_key"
    ]
].sort_values(
    "first_publish_year"
)

,title,authors,first_publish_year,publish_dates,openlibrary_key


In [53]:
print(
    "Minimum rating:",
    openlibrary_search_unique[
        "ratings_average"
    ].min()
)

print(
    "Maximum rating:",
    openlibrary_search_unique[
        "ratings_average"
    ].max()
)

print(
    "Records with ratings:",
    openlibrary_search_unique[
        "ratings_average"
    ].notna().sum()
)

Minimum rating: 0.0
Maximum rating: 5.0
Records with ratings: 282


In [54]:
invalid_ratings = (
    openlibrary_search_unique.loc[
        (
            openlibrary_search_unique[
                "ratings_average"
            ] < 0
        )
        |
        (
            openlibrary_search_unique[
                "ratings_average"
            ] > 5
        )
    ]
)

print(
    "Invalid rating values:",
    len(invalid_ratings)
)

Invalid rating values: 0


In [55]:
rating_status = pd.Series(
    np.select(
        [
            openlibrary_search_unique[
                "ratings_average"
            ].isna(),

            (
                openlibrary_search_unique[
                    "ratings_count"
                ] == 0
            )
        ],
        [
            "Missing rating data",
            "Zero recorded ratings"
        ],
        default="Has rating data"
    ),
    index=openlibrary_search_unique.index,
    name="rating_status"
)

rating_status.value_counts()

rating_status
Missing rating data      668
Has rating data          268
Zero recorded ratings     14
Name: count, dtype: int64

In [56]:
openlibrary_search_unique[
    "rating_status"
] = rating_status

In [57]:
rating_count_columns = [
    "ratings_count_1",
    "ratings_count_2",
    "ratings_count_3",
    "ratings_count_4",
    "ratings_count_5"
]

openlibrary_search_unique[
    "rating_distribution_total"
] = (
    openlibrary_search_unique[
        rating_count_columns
    ]
    .sum(
        axis=1,
        min_count=1
    )
)

In [58]:
rating_count_consistent = (
    openlibrary_search_unique[
        "ratings_count"
    ]
    ==
    openlibrary_search_unique[
        "rating_distribution_total"
    ]
)

comparable_ratings = (
    openlibrary_search_unique[
        "ratings_count"
    ].notna()
    &
    openlibrary_search_unique[
        "rating_distribution_total"
    ].notna()
)

print(
    "Comparable records:",
    comparable_ratings.sum()
)

print(
    "Consistent rating totals:",
    (
        comparable_ratings
        & rating_count_consistent
    ).sum()
)

print(
    "Inconsistent rating totals:",
    (
        comparable_ratings
        & ~rating_count_consistent
    ).sum()
)

Comparable records: 282
Consistent rating totals: 282
Inconsistent rating totals: 0


In [59]:
openlibrary_search_unique.loc[
    comparable_ratings
    & ~rating_count_consistent,
    [
        "title",
        "ratings_average",
        "ratings_count",
        "ratings_count_1",
        "ratings_count_2",
        "ratings_count_3",
        "ratings_count_4",
        "ratings_count_5",
        "rating_distribution_total"
    ]
].head(20)

,title,ratings_average,ratings_count,ratings_count_1,ratings_count_2,ratings_count_3,ratings_count_4,ratings_count_5,rating_distribution_total


## 34. Open Library Search — Validation Results

Following work-level consolidation, the Open Library Search dataset contains 950 unique works.

### Query Overlap

Some works were retrieved through multiple leadership and management search queries.

Rather than retaining these as duplicate book records, the search-query memberships were consolidated into:

- `collection_queries`
- `query_match_count`

This preserves information about thematic retrieval overlap without artificially duplicating books in later analyses.

The highest observed query-match count was six.

`query_match_count` represents retrieval overlap within this project's search design. It must not be interpreted directly as book quality, popularity, or importance.

### Publication Year Validation

Among the 950 unique works:

- 947 contain a first-publication year;
- 3 have no first-publication year;
- the earliest observed year is 1900;
- the latest observed year is 2025;
- no publication years fall outside the validation range.

The three missing publication years will remain missing unless they can later be resolved from another collected source.

### Rating Validation

Of the 950 unique works:

- 282 contain a rating value;
- 268 contain rating activity;
- 14 explicitly contain zero recorded ratings;
- 668 have missing rating information.

Observed average ratings range from 0.0 to 5.0.

No values fall outside the expected 0–5 rating range.

Missing rating information is not interpreted as a zero rating.

### Rating Distribution

For records where rating totals and star-level rating counts are available, the star-level distributions were checked against the reported total rating count.

No inconsistencies were identified in the validation output.

In [60]:
openlibrary_search_unique[
    "first_publish_year"
] = (
    openlibrary_search_unique[
        "first_publish_year"
    ].astype("Int64")
)

In [61]:
print(
    openlibrary_search_unique[
        "first_publish_year"
    ].dtype
)

print(
    openlibrary_search_unique[
        "first_publish_year"
    ].isna().sum()
)

Int64
3


In [62]:
count_columns = [
    "edition_count",
    "ratings_count",
    "ratings_count_1",
    "ratings_count_2",
    "ratings_count_3",
    "ratings_count_4",
    "ratings_count_5",
    "want_to_read_count",
    "currently_reading_count",
    "already_read_count",
    "cover_id",
    "query_match_count",
    "rating_distribution_total"
]

for column in count_columns:

    openlibrary_search_unique[
        column
    ] = (
        openlibrary_search_unique[
            column
        ].astype("Int64")
    )

In [63]:
openlibrary_search_unique[
    count_columns
].dtypes

edition_count                Int64
ratings_count                Int64
ratings_count_1              Int64
ratings_count_2              Int64
ratings_count_3              Int64
ratings_count_4              Int64
ratings_count_5              Int64
want_to_read_count           Int64
currently_reading_count      Int64
already_read_count           Int64
cover_id                     Int64
query_match_count            Int64
rating_distribution_total    Int64
dtype: object

In [64]:
print(
    "Rows:",
    len(openlibrary_search_unique)
)

print(
    "Columns:",
    openlibrary_search_unique.shape[1]
)

print(
    "Unique works:",
    openlibrary_search_unique[
        "openlibrary_key"
    ].nunique()
)

print(
    "Duplicate work keys:",
    openlibrary_search_unique[
        "openlibrary_key"
    ].duplicated().sum()
)

print(
    "Missing titles:",
    openlibrary_search_unique[
        "title"
    ].isna().sum()
)

print(
    "Missing authors:",
    openlibrary_search_unique[
        "authors"
    ].isna().sum()
)

Rows: 950
Columns: 34
Unique works: 950
Duplicate work keys: 0
Missing titles: 0
Missing authors: 8


# Open Library Work Enrichment Cleaning

The Open Library Work API enrichment dataset contains one record for each of the 950 unique Open Library works identified during Search API collection.

This dataset provides supplementary information including:

- descriptions;
- first sentences;
- subjects;
- geographic subjects;
- people;
- time-period subjects;
- excerpts;
- Library of Congress classifications;
- Dewey Decimal classifications;
- work-level cover identifiers.

The enrichment dataset will remain separate from the Search dataset during cleaning.

The two Open Library datasets will be combined only after their identifiers and field structures have been validated.

In [65]:
search_keys = set(
    openlibrary_search_unique[
        "openlibrary_key"
    ]
)

work_keys = set(
    openlibrary_work_clean[
        "openlibrary_key"
    ]
)

print(
    "Unique Search keys:",
    len(search_keys)
)

print(
    "Unique Work keys:",
    len(work_keys)
)

print(
    "Keys in Search but not Work:",
    len(
        search_keys - work_keys
    )
)

print(
    "Keys in Work but not Search:",
    len(
        work_keys - search_keys
    )
)

Unique Search keys: 950
Unique Work keys: 950
Keys in Search but not Work: 0
Keys in Work but not Search: 0


In [66]:
openlibrary_work_clean[
    "work_status_code"
].value_counts(
    dropna=False
)

work_status_code
200    950
Name: count, dtype: int64

In [67]:
work_list_columns = [
    "work_subjects",
    "subject_places",
    "subject_people",
    "subject_times",
    "excerpts",
    "lc_classifications",
    "dewey_number",
    "work_covers"
]

for column in work_list_columns:

    print(
        f"\n--- {column} ---"
    )

    print(
        openlibrary_work_clean[
            column
        ]
        .dropna()
        .head(3)
        .tolist()
    )


--- work_subjects ---
["['Leadership', 'Psychological aspects of Success', 'Success', 'Psychological aspects', 'Commerce', 'Success in business', 'Aptitude pour la direction', 'Achievement', 'Success--']", "['Organisation', 'Prise de décision', 'Entscheidungsfindung', 'Leadership', 'Organizational sociology', 'Organization', 'Decision making', 'Management', 'Entscheidungstheorie', 'Politische Wissenschaft', 'Economic development', 'Fu\\x98hrung', 'Prise de decision', '85.08 organizational sociology and psychology', 'Führung', 'Organisationsverhalten', 'Leiderschap', 'Armies', 'Executive ability', 'Organizational Decision Making', 'Manuel', 'Organisation (organisme)', 'Ledarskap', 'Organisationssociologi', 'Organisationsteori', 'Organisationsutveckling', 'Beslutsfattande', 'Organization theory']", "['Leadership']"]

--- subject_places ---
["['United States']", "['United States']", "['New York', 'New York (N.Y.)', 'New York (State)', 'United States', 'New York (État)', 'États-Unis']"]



In [69]:
for column in work_list_columns:

    non_missing = (
        openlibrary_work_clean[
            column
        ].dropna()
    )

    if len(non_missing) == 0:
        continue

    sample = non_missing.iloc[0]

    parsed_sample = (
        parse_serialized_list(
            sample
        )
    )

    print(
        f"\n{column}"
    )

    print(
        "Before:",
        sample
    )

    print(
        "After:",
        parsed_sample
    )

    print(
        "Type:",
        type(
            parsed_sample
        ).__name__
    )


work_subjects
Before: ['Leadership', 'Psychological aspects of Success', 'Success', 'Psychological aspects', 'Commerce', 'Success in business', 'Aptitude pour la direction', 'Achievement', 'Success--']
After: ['Leadership', 'Psychological aspects of Success', 'Success', 'Psychological aspects', 'Commerce', 'Success in business', 'Aptitude pour la direction', 'Achievement', 'Success--']
Type: list

subject_places
Before: ['United States']
After: ['United States']
Type: list

subject_people
Before: ['James B. Comey Jr. (1960-)', 'George W. Bush (1946-)', 'Donald Trump (1946-)']
After: ['James B. Comey Jr. (1960-)', 'George W. Bush (1946-)', 'Donald Trump (1946-)']
Type: list

subject_times
Before: ['2017-', '2001-2009', '2009-2017', '2016', '21st century']
After: ['2017-', '2001-2009', '2009-2017', '2016', '21st century']
Type: list

excerpts
Before: ['I HAVE LONG ADVOCATED a natural, gradual, day-by-day, step-by-step, sequential approach to personal development.']
After: ['I HAVE LONG 

In [70]:
for column in work_list_columns:

    openlibrary_work_clean[
        column
    ] = (
        openlibrary_work_clean[
            column
        ].apply(
            parse_serialized_list
        )
    )

In [71]:
work_parsing_audit = []

for column in work_list_columns:

    non_missing = (
        openlibrary_work_clean[
            column
        ].dropna()
    )

    list_count = (
        non_missing.apply(
            lambda x:
                isinstance(x, list)
        ).sum()
    )

    work_parsing_audit.append({
        "column": column,
        "non_missing_values": len(non_missing),
        "parsed_as_list": list_count,
        "not_parsed_as_list":
            len(non_missing) - list_count
    })

work_parsing_audit_df = pd.DataFrame(
    work_parsing_audit
)

work_parsing_audit_df

,column,non_missing_values,parsed_as_list,not_parsed_as_list
0,work_subjects,826,826,0
1,subject_places,56,56,0
2,subject_people,13,13,0
3,subject_times,9,9,0
4,excerpts,950,950,0
5,lc_classifications,85,85,0
6,dewey_number,155,155,0
7,work_covers,743,743,0


In [72]:
work_list_availability = []

for column in work_list_columns:

    with_values = (
        openlibrary_work_clean[
            column
        ].apply(
            lambda x:
                isinstance(x, list)
                and len(x) > 0
        ).sum()
    )

    empty_lists = (
        openlibrary_work_clean[
            column
        ].apply(
            lambda x:
                isinstance(x, list)
                and len(x) == 0
        ).sum()
    )

    missing_values = (
        openlibrary_work_clean[
            column
        ].isna().sum()
    )

    work_list_availability.append({
        "column": column,
        "with_values": with_values,
        "empty_lists": empty_lists,
        "missing_values": missing_values,
        "coverage_pct":
            round(
                with_values
                / len(openlibrary_work_clean)
                * 100,
                2
            )
    })

work_list_availability_df = pd.DataFrame(
    work_list_availability
)

work_list_availability_df

,column,with_values,empty_lists,missing_values,coverage_pct
0,work_subjects,826,0,124,86.95
1,subject_places,56,0,894,5.89
2,subject_people,13,0,937,1.37
3,subject_times,9,0,941,0.95
4,excerpts,23,927,0,2.42
5,lc_classifications,85,0,865,8.95
6,dewey_number,155,0,795,16.32
7,work_covers,743,0,207,78.21


In [74]:
for column in work_text_list_columns:

    openlibrary_work_clean[
        column
    ] = (
        openlibrary_work_clean[
            column
        ]
        .apply(
            clean_string_list
        )
        .apply(
            remove_list_duplicates
        )
    )

In [75]:
openlibrary_work_clean[
    "work_covers"
] = (
    openlibrary_work_clean[
        "work_covers"
    ].apply(
        remove_list_duplicates
    )
)

In [76]:
work_scalar_text_columns = [
    "openlibrary_key",
    "title",
    "description",
    "first_sentence"
]

for column in work_scalar_text_columns:

    openlibrary_work_clean[
        column
    ] = (
        openlibrary_work_clean[
            column
        ].apply(
            clean_text_value
        )
    )

In [77]:
work_cleaned_availability = []

for column in work_list_columns:

    with_values = (
        openlibrary_work_clean[
            column
        ].apply(
            lambda x:
                isinstance(x, list)
                and len(x) > 0
        ).sum()
    )

    work_cleaned_availability.append({
        "field": column,
        "records_with_values":
            with_values,
        "coverage_pct":
            round(
                with_values
                / len(openlibrary_work_clean)
                * 100,
                2
            )
    })

pd.DataFrame(
    work_cleaned_availability
).sort_values(
    "coverage_pct",
    ascending=False
)

,field,records_with_values,coverage_pct
0,work_subjects,826,86.95
7,work_covers,743,78.21
6,dewey_number,155,16.32
5,lc_classifications,85,8.95
1,subject_places,56,5.89
4,excerpts,23,2.42
2,subject_people,13,1.37
3,subject_times,9,0.95


In [78]:
scalar_coverage = pd.DataFrame({
    "field": [
        "description",
        "first_sentence"
    ],

    "records_with_values": [
        openlibrary_work_clean[
            "description"
        ].notna().sum(),

        openlibrary_work_clean[
            "first_sentence"
        ].notna().sum()
    ]
})

scalar_coverage[
    "coverage_pct"
] = (
    scalar_coverage[
        "records_with_values"
    ]
    / len(openlibrary_work_clean)
    * 100
).round(2)

scalar_coverage

,field,records_with_values,coverage_pct
0,description,192,20.21
1,first_sentence,32,3.37


In [79]:
title_comparison = (
    openlibrary_search_unique[
        [
            "openlibrary_key",
            "title"
        ]
    ]
    .merge(
        openlibrary_work_clean[
            [
                "openlibrary_key",
                "title"
            ]
        ],
        on="openlibrary_key",
        how="inner",
        suffixes=(
            "_search",
            "_work"
        )
    )
)

In [80]:
def normalize_title_for_comparison(value):

    if pd.isna(value):
        return ""

    value = str(value).lower().strip()

    value = re.sub(
        r"\s+",
        " ",
        value
    )

    return value

In [81]:
title_comparison[
    "title_match"
] = (
    title_comparison[
        "title_search"
    ].apply(
        normalize_title_for_comparison
    )
    ==
    title_comparison[
        "title_work"
    ].apply(
        normalize_title_for_comparison
    )
)

In [82]:
print(
    "Titles compared:",
    len(title_comparison)
)

print(
    "Matching titles:",
    title_comparison[
        "title_match"
    ].sum()
)

print(
    "Different titles:",
    (
        ~title_comparison[
            "title_match"
        ]
    ).sum()
)

Titles compared: 950
Matching titles: 950
Different titles: 0


In [83]:
title_comparison.loc[
    ~title_comparison[
        "title_match"
    ],
    [
        "openlibrary_key",
        "title_search",
        "title_work"
    ]
].head(20)

,openlibrary_key,title_search,title_work


In [84]:
openlibrary_work_clean[
    "description_length"
] = (
    openlibrary_work_clean[
        "description"
    ]
    .fillna("")
    .str.len()
)

In [85]:
openlibrary_work_clean.loc[
    openlibrary_work_clean[
        "description"
    ].notna(),
    "description_length"
].describe()

count     192.000000
mean      718.656250
std       618.734629
min         5.000000
25%       257.500000
50%       628.500000
75%       963.250000
max      3247.000000
Name: description_length, dtype: float64

In [86]:
openlibrary_work_clean.loc[
    openlibrary_work_clean[
        "description"
    ].notna(),
    [
        "title",
        "description",
        "description_length"
    ]
].head(10)

,title,description,description_length
0,Principle-Centered Leadership,How do we as individuals and organizations sur...,1104
6,Leadership,"""The Third Edition of this bestselling text re...",559
8,A Higher Loyalty,The former FBI director shares his experiences...,180
9,Leadership and Self Deception,"This phenomenal bestseller * over 700,000 copi...",339
12,Lincoln on Leadership,A guide to business leadership based on the st...,217
13,Leadership,"A systematic study, ranging from the salons of...",243
15,The leadership challenge,"When it was initially written in 1987, few cou...",964
16,ORGANIZATIONAL CULTURE AND LEADERSHIP,Organizational culture refers to culture in an...,758
18,Leadership In Turbulent Times,In this culmination of five decades of acclaim...,1444
20,Transformational Leadership,"""Transformational Leadership, Second Edition i...",456


In [87]:
openlibrary_work_clean[
    "work_subject_count"
] = (
    openlibrary_work_clean[
        "work_subjects"
    ].apply(
        lambda x:
            len(x)
            if isinstance(x, list)
            else 0
    )
)

In [88]:
openlibrary_work_clean[
    "work_subject_count"
].describe()

count    950.000000
mean       6.298947
std        7.221404
min        0.000000
25%        1.000000
50%        4.000000
75%        9.000000
max       59.000000
Name: work_subject_count, dtype: float64

In [89]:
openlibrary_work_clean[
    [
        "title",
        "work_subject_count",
        "work_subjects"
    ]
].sort_values(
    "work_subject_count",
    ascending=False
).head(10)

,title,work_subject_count,work_subjects
858,Working with Emotional Intelligence,59,"[Management, Emotions, Développement de la per..."
332,The 7 Habits of Highly Effective People,59,"[Success- Psychological aspects, Character, Su..."
29,Reframing Organizations,48,"[Leadership, Management, Organizational behavi..."
470,Rework,41,"[Business, Nonfiction, New York Times bestsell..."
149,Team of Rivals,40,"[Union Army, Thirteenth Amendment, assassinati..."
855,Emotional Intelligence,38,"[Emotions and cognition, Intellect, Success, p..."
914,The Innovator's Dilemma,38,"[Industrial management, Disruptive technologie..."
931,Innovation and Entrepreneurship,34,"[Empreses, Emprenedoria, Empreses petites i mi..."
18,Leadership In Turbulent Times,33,"[Politics and government, Political culture, P..."
188,Team of Teams,33,"[United States, Reorganization, Biography, Org..."


In [90]:
print(
    "Rows:",
    len(openlibrary_work_clean)
)

print(
    "Unique work keys:",
    openlibrary_work_clean[
        "openlibrary_key"
    ].nunique()
)

print(
    "Duplicate work keys:",
    openlibrary_work_clean[
        "openlibrary_key"
    ].duplicated().sum()
)

print(
    "Missing keys:",
    openlibrary_work_clean[
        "openlibrary_key"
    ].isna().sum()
)

print(
    "Missing titles:",
    openlibrary_work_clean[
        "title"
    ].isna().sum()
)

print(
    "Successful API responses:",
    (
        openlibrary_work_clean[
            "work_status_code"
        ] == 200
    ).sum()
)

Rows: 950
Unique work keys: 950
Duplicate work keys: 0
Missing keys: 0
Missing titles: 0
Successful API responses: 950


## Open Library Work Enrichment — Cleaning Interpretation

The Open Library Search and Work datasets demonstrate complete identifier alignment.

All 950 unique Open Library works collected through the Search API are represented in the Work enrichment dataset, and all Work API requests returned HTTP status 200.

List-based metadata fields were reconstructed from their serialized CSV representations before evaluating data availability.

This distinction is important because a serialized empty list (`[]`) is technically a non-missing CSV value but does not contain usable metadata.

Consequently, enrichment coverage is evaluated using populated lists rather than relying solely on conventional null-value counts.

Text normalization was deliberately conservative. Descriptions and other natural-language fields were not lowercased, stemmed, lemmatized, or otherwise transformed for machine learning during data cleaning.

NLP-specific preprocessing will be performed later in the NLP notebook so that the cleaned dataset retains human-readable source text.

The Open Library work identifier remains the primary relationship between Search and Work datasets. Title comparisons are used only as a secondary quality-control check.

# LeadershipNow / LeaderShop Data Cleaning

The LeadershipNow dataset contains 1,124 raw book records collected from the complete 2022–2025 release archives.

Unlike the Open Library dataset, the LeadershipNow source contains edition-oriented bibliographic information including:

- title;
- subtitle;
- author;
- book format;
- page count embedded within the format field;
- ISBN;
- publisher;
- publication date;
- cover information;
- external book link;
- release-page provenance.

The cleaning process will preserve the raw source fields while deriving standardized variables for later integration and analysis.

Particular attention will be given to:

1. title, subtitle, and author normalization;
2. format and page-count extraction;
3. ISBN normalization and check-digit validation;
4. investigation of unusual ISBN lengths;
5. publication-date parsing;
6. duplicate ISBN investigation;
7. preservation of release-page provenance.

No questionable ISBN or duplicate record will be automatically removed without validation.

In [91]:
leadershipnow_clean = (
    leadershipnow_raw.copy()
)

print(
    leadershipnow_clean.shape
)

(1124, 15)


In [92]:
leadershipnow_clean[
    [
        "title",
        "subtitle",
        "author",
        "format_raw",
        "isbn",
        "publisher",
        "publication_date_raw"
    ]
].head(15)

,title,subtitle,author,format_raw,isbn,publisher,publication_date_raw
0,The Seismic Shift in You,: The Seven Necessary Shifts to Create Connect...,Michelle Johnston and Marshall Goldsmith,"Hardcover, 152 pages",9798891386211,100 Coaches Publishing,"December 2, 2025"
1,The View from Ninety,": Reflections on Living a Long, Contented Life",Charles Handy,"Hardcover, 224 pages",9781529154801,Hutchinson Heinemann,"December 2, 2025"
2,In the Arena,": Theodore Roosevelt in War, Peace, and Revolu...",David S. Brown,"Hardcover, 496 pages",9781668204191,Scribner,"December 2, 2025"
3,You to the Power of Two,: Redefining Human Potential in the Age of Ide...,Joseph Bradley and Don Tapscott,"Hardcover, 400 pages",9781637747841,BenBella Books,"December 2, 2025"
4,One Move Makes All the Difference,: How to discover your power and transform you...,Martin R. Mendelson,"Paperback, 200 pages",9781636987897,Morgan James Publishing,"December 2, 2025"
5,How to be a Friend (In an Unfriendly World),: Lessons on Connection,Barnet Bain,"Hardcover, 208 pages",9781394388868,Wiley,"December 9, 2025"
6,Be Astonishing,: How to Transform Ordinary Influence into Las...,Sam Silverstein and Allison Silverstein,"Paperback, 176 pages",9781640955837,Sound Wisdom,"December 9, 2025"
7,Monster Transformation,": Conquer Your Digital Fears, Be AI Ready, and...","Ari Lightman, Rafeh Masood and Gary Hirsch","Hardcover, 252 pages",9781394313709,Wiley,"December 9, 2025"
8,How to Make a Few More Billion Dollars,NaN,Brad Jacobs,"Hardcover, 296 pages",9798886454659,Greenleaf Book Group Press,"December 9, 2025"
9,Human Edge in the AI Age,: Eight Timeless Mantras For Success,Nitin Seth,"Hardcover, 496 pages",9798216391746,Bloomsbury Academic,"December 11, 2025"


In [93]:
leadershipnow_clean[
    "format_raw"
].value_counts().head(25)

format_raw
Hardcover, 256 pages    133
Hardcover, 240 pages     82
Hardcover, 272 pages     79
Hardcover, 288 pages     75
Hardcover, 224 pages     58
Hardcover, 304 pages     57
Hardcover, 320 pages     49
Hardcover, 352 pages     28
Hardcover, 208 pages     26
Hardcover, 336 pages     23
Paperback, 240 pages     22
Hardcover, 368 pages     19
Hardcover, 400 pages     16
Hardcover, 192 pages     16
Paperback, 256 pages     16
Hardcover, 384 pages     15
Hardcover, 176 pages     14
Paperback, 192 pages     13
Paperback, 176 pages     11
Hardcover, 264 pages     11
Hardcover, 496 pages     10
Hardcover, 200 pages     10
Hardcover, 248 pages     10
Paperback, 208 pages     10
Hardcover, 448 pages      9
Name: count, dtype: int64

In [94]:
leadershipnow_clean[
    "format_raw"
].sample(
        20,
        random_state=42
    ).tolist()

['Hardcover,\xa0384 pages',
 'Hardcover,\xa0304 pages',
 'Hardcover,\xa0288 pages',
 'Hardcover,\xa0256 pages',
 'Hardcover,\xa0184 pages',
 'Hardcover,\xa0336 pages',
 'Hardcover,\xa0304 pages',
 'Paperback,\xa0141 pages',
 'Hardcover,\xa0256 pages',
 'Paperback,\xa0128 pages',
 'Hardcover,\xa0236 pages',
 'Paperback,\xa0160 pages',
 'Hardcover,\xa0256 pages',
 'Hardcover,\xa0280 pages',
 'Hardcover,\xa0288 pages',
 'Paperback,\xa0226 pages',
 'Paperback,\xa0208 pages',
 'Hardcover,\xa0224 pages',
 'Hardcover,\xa0256 pages',
 'Hardcover,\xa0336 pages']

In [95]:
leadershipnow_clean[
    "isbn_normalized"
] = (
    leadershipnow_clean[
        "isbn"
    ]
    .astype(str)
    .str.upper()
    .str.replace(
        r"[^0-9X]",
        "",
        regex=True
    )
)

In [96]:
leadershipnow_clean[
    "isbn_length"
] = (
    leadershipnow_clean[
        "isbn_normalized"
    ].str.len()
)

In [97]:
leadershipnow_clean[
    "isbn_length"
].value_counts().sort_index()

isbn_length
12       2
13    1121
14       1
Name: count, dtype: int64

In [98]:
isbn_length_anomalies = (
    leadershipnow_clean.loc[
        leadershipnow_clean[
            "isbn_length"
        ] != 13,
        [
            "scrape_id",
            "title",
            "subtitle",
            "author",
            "isbn",
            "isbn_normalized",
            "isbn_length",
            "publisher",
            "publication_date_raw",
            "release_page"
        ]
    ]
)

isbn_length_anomalies

,scrape_id,title,subtitle,author,isbn,isbn_normalized,isbn_length,publisher,publication_date_raw,release_page
146,147,The New Emotional Intelligence,NaN,Travis Bradberry,979218589660,979218589660,12,Bruyere Publishing,"May 13, 2025",2025 Releases
455,456,The Energy of Success,": Power Up Your Productivity, Transform Your H...",Rebecca Ahmed,97811394245475,97811394245475,14,Wiley,"April 23, 2024",2024 Releases
878,879,When McKinsey Comes to Town,: The Hidden Influence of the World's Most Pow...,Walt Bogdanich and Michael Forsythe,978385546232,978385546232,12,Doubleday,"October 4, 2022",2022 Releases


In [99]:
def validate_isbn13(value):
    """
    Validate an ISBN-13 using its check digit.

    Returns:
        True  -> structurally valid ISBN-13
        False -> 13 characters but invalid check digit
        <NA>  -> cannot be evaluated as ISBN-13
    """

    if pd.isna(value):
        return pd.NA

    value = str(value).strip()

    if not re.fullmatch(
        r"\d{13}",
        value
    ):
        return pd.NA

    digits = [
        int(digit)
        for digit in value
    ]

    weighted_sum = sum(
        digit * (
            1 if index % 2 == 0
            else 3
        )
        for index, digit
        in enumerate(
            digits[:12]
        )
    )

    expected_check_digit = (
        10 - (
            weighted_sum % 10
        )
    ) % 10

    return (
        expected_check_digit
        == digits[12]
    )

In [100]:
leadershipnow_clean[
    "isbn13_valid"
] = (
    leadershipnow_clean[
        "isbn_normalized"
    ].apply(
        validate_isbn13
    ).astype("boolean")
)

In [101]:
leadershipnow_clean[
    "isbn13_valid"
].value_counts(
    dropna=False
)

isbn13_valid
True     1061
False      58
<NA>        5
Name: count, dtype: Int64

In [102]:
invalid_isbn13 = (
    leadershipnow_clean.loc[
        leadershipnow_clean[
            "isbn13_valid"
        ] == False,
        [
            "title",
            "author",
            "isbn",
            "isbn_normalized",
            "publisher",
            "publication_date_raw",
            "release_page"
        ]
    ]
)

print(
    "Invalid 13-digit ISBNs:",
    len(invalid_isbn13)
)

invalid_isbn13.head(20)

Invalid 13-digit ISBNs: 58


,title,author,isbn,isbn_normalized,publisher,publication_date_raw,release_page
278,The Purposeful Decision Maker,Pádraig Ó Céidigh,9788891381322,9788891381322,Amplify Publishing,"November 12, 2024",2024 Releases
302,The Seven Frequencies of Communication,Erwin Raphael McManus,9788991045612,9788991045612,Erwin McManus Publishing,"October 15, 2024",2024 Releases
305,Crisis Capable,Fabiana Lacerca-Allen,9708891880115,9708891880115,Advantage Media Group,"October 15, 2024",2024 Releases
312,Burned Out to Lit Up,Cara E. Houser,9788988925200,9788988925200,Parliament Press,"October 24, 2024",2024 Releases
318,The Reset Mindset,Penny Zenker,9788891382299,9788891382299,Amplify Publishing,"September 3, 2024",2024 Releases
348,AI Snake Oil,Arvind Narayanan and Sayash Kapoor,9790691249131,9790691249131,Princeton University Press,"September 24, 2024",2024 Releases
356,Leading What Matters Most,Phil Geldart,9788887502472,9788887502472,ForbesBooks,"August 13, 2024",2024 Releases
376,Front-Line Leadership,Patrick Nelson,9791394240753,9791394240753,Wiley,"July 11, 2024",2024 Releases
383,The Turnaround Leader,Wes Wheeler,9788891880344,9788891880344,Advantage Media Group,"July 30, 2024",2024 Releases
385,The Soil of Leadership,Britt Yamamoto,9788891380554,9788891380554,Amplify Publishing,"July 30, 2024",2024 Releases


In [103]:
leadershipnow_clean[
    "isbn_prefix"
] = (
    leadershipnow_clean[
        "isbn_normalized"
    ].str[:3]
)

In [104]:
leadershipnow_clean[
    "isbn_prefix"
].value_counts(
    dropna=False
)

isbn_prefix
978    1034
979      89
970       1
Name: count, dtype: int64

In [105]:
leadershipnow_clean.loc[
    (
        leadershipnow_clean[
            "isbn_length"
        ] == 13
    )
    &
    (
        ~leadershipnow_clean[
            "isbn_prefix"
        ].isin(
            ["978", "979"]
        )
    ),
    [
        "title",
        "author",
        "isbn_normalized",
        "isbn_prefix"
    ]
]

,title,author,isbn_normalized,isbn_prefix
305,Crisis Capable,Fabiana Lacerca-Allen,9708891880115,970


In [106]:
leadershipnow_clean[
    "title"
] = (
    leadershipnow_clean[
        "title"
    ].apply(
        clean_text_value
    )
)

In [107]:
print(
    "Missing titles:",
    leadershipnow_clean[
        "title"
    ].isna().sum()
)

print(
    "Unique titles:",
    leadershipnow_clean[
        "title"
    ].nunique()
)

Missing titles: 0
Unique titles: 1114


In [108]:
leadershipnow_clean.loc[
    leadershipnow_clean[
        "subtitle"
    ].notna(),
    "subtitle"
].head(20).tolist()

[': The Seven Necessary Shifts to Create Connection and Drive Results',
 ': Reflections on Living a Long, Contented Life',
 ': Theodore Roosevelt in War, Peace, and Revolution',
 ': Redefining Human Potential in the Age of Identic AI',
 ': How to discover your power and transform your life',
 ': Lessons on Connection',
 ': How to Transform Ordinary Influence into Lasting Significance',
 ': Conquer Your Digital Fears, Be AI Ready, and Focus on What Matters to Your Organization',
 ': Eight Timeless Mantras For Success',
 ': How to Lead with Your Unique Value',
 ': A Guide to Thriving in a World of Continuous Transformation',
 ': How To Achieve Transformational Change, Scale, And Growth Simultaneously',
 ': Embodying Executive Presence to Lead with Impact',
 ': Rejecting Competition to Unlock Success',
 ': The Secret Brand Strategy for Creating Competitive Advantage',
 ': Five Communication Habits For Limitless Influence and Business Success Hardcover \x96 November 4, 2025\r\nby Ravi Raja

In [109]:
def clean_subtitle(value):

    value = clean_text_value(
        value
    )

    if pd.isna(value):
        return np.nan

    value = re.sub(
        r"^\s*:\s*",
        "",
        value
    )

    value = value.strip()

    return (
        value
        if value
        else np.nan
    )

In [110]:
leadershipnow_clean[
    "subtitle"
] = (
    leadershipnow_clean[
        "subtitle"
    ].apply(
        clean_subtitle
    )
)

In [111]:
print(
    "Missing subtitles:",
    leadershipnow_clean[
        "subtitle"
    ].isna().sum()
)

leadershipnow_clean[
    "subtitle"
].head(20).tolist()

Missing subtitles: 43


['The Seven Necessary Shifts to Create Connection and Drive Results',
 'Reflections on Living a Long, Contented Life',
 'Theodore Roosevelt in War, Peace, and Revolution',
 'Redefining Human Potential in the Age of Identic AI',
 'How to discover your power and transform your life',
 'Lessons on Connection',
 'How to Transform Ordinary Influence into Lasting Significance',
 'Conquer Your Digital Fears, Be AI Ready, and Focus on What Matters to Your Organization',
 nan,
 'Eight Timeless Mantras For Success',
 'How to Lead with Your Unique Value',
 nan,
 'A Guide to Thriving in a World of Continuous Transformation',
 'How To Achieve Transformational Change, Scale, And Growth Simultaneously',
 'Embodying Executive Presence to Lead with Impact',
 'Rejecting Competition to Unlock Success',
 'The Secret Brand Strategy for Creating Competitive Advantage',
 'Five Communication Habits For Limitless Influence and Business Success Hardcover \x96 November 4, 2025 by Ravi Rajani (',
 'How to Focus o

In [112]:
for column in [
    "author",
    "publisher"
]:

    leadershipnow_clean[
        column
    ] = (
        leadershipnow_clean[
            column
        ].apply(
            clean_text_value
        )
    )

In [113]:
print(
    "Missing authors:",
    leadershipnow_clean[
        "author"
    ].isna().sum()
)

print(
    "Missing publishers:",
    leadershipnow_clean[
        "publisher"
    ].isna().sum()
)

Missing authors: 0
Missing publishers: 0


In [114]:
format_samples = (
    leadershipnow_clean[
        "format_raw"
    ]
    .dropna()
    .drop_duplicates()
)

print(
    "Unique format strings:",
    len(format_samples)
)

format_samples.head(50).tolist()

Unique format strings: 196


['Hardcover,\xa0152 pages',
 'Hardcover,\xa0224 pages',
 'Hardcover,\xa0496 pages',
 'Hardcover,\xa0400 pages',
 'Paperback,\xa0200 pages',
 'Hardcover,\xa0208 pages',
 'Paperback,\xa0176 pages',
 'Hardcover,\xa0252 pages',
 'Hardcover,\xa0296 pages',
 'Hardcover,\xa0110 pages',
 'Hardcover,\xa0240 pages',
 'Hardcover,\xa0368 pages',
 'Hardcover,\xa0280 pages',
 'Hardcover,\xa0140 pages',
 'Hardcover,\xa0320 pages',
 'Paperback,\xa0160 pages',
 'Paperback,\xa0168 pages',
 'Hardcover,\xa0232 pages',
 'Hardcover,\xa0352 pages',
 'Hardcover,\xa0192 pages',
 'Hardcover,\xa0416 pages',
 'Hardcover,\xa0256 pages',
 'Hardcover,\xa0168 pages',
 'Paperback,\xa0192 pages',
 'Hardcover,\xa0304 pages',
 'Hardcover,\xa0184 pages',
 'Hardcover,\xa0200 pages',
 'Paperback,\xa0256 pages',
 'Hardcover,\xa0288 pages',
 'Hardcover,\xa0384 pages',
 'Hardcover,\xa0448 pages',
 'Paperback,\xa0166 pages',
 'Hardcover,\xa0210 pages',
 'Hardcover,\xa0336 pages',
 'Hardcover,\xa0264 pages',
 'Hardcover,\xa0462 

In [115]:
contains_pages = (
    leadershipnow_clean[
        "format_raw"
    ]
    .str.contains(
        r"\bpages?\b",
        case=False,
        na=False
    )
)

print(
    "Records containing page information:",
    contains_pages.sum()
)

print(
    "Records without explicit page information:",
    (~contains_pages).sum()
)

Records containing page information: 1124
Records without explicit page information: 0


In [116]:
leadershipnow_clean.loc[
    ~contains_pages,
    [
        "title",
        "format_raw"
    ]
].head(30)

,title,format_raw


## 63. LeadershipNow Subtitle Quality Audit

Initial subtitle normalization successfully removed the source's leading colon formatting.

However, inspection identified at least one subtitle containing text that appears to belong to adjacent bibliographic metadata, including format, publication date, and author information.

Rather than automatically deleting text using assumptions about subtitle structure, potentially contaminated subtitles will be identified and inspected before correction.

In [117]:
suspicious_subtitle_pattern = (
    r"\b("
    r"Hardcover|"
    r"Paperback|"
    r"pages?|"
    r"January|February|March|April|May|June|"
    r"July|August|September|October|November|December"
    r")\b"
)

suspicious_subtitles = (
    leadershipnow_clean.loc[
        leadershipnow_clean[
            "subtitle"
        ].str.contains(
            suspicious_subtitle_pattern,
            case=False,
            na=False,
            regex=True
        ),
        [
            "scrape_id",
            "title",
            "subtitle",
            "author",
            "format_raw",
            "publication_date_raw",
            "isbn",
            "release_page"
        ]
    ]
)

print(
    "Potentially suspicious subtitles:",
    len(suspicious_subtitles)
)

suspicious_subtitles

Potentially suspicious subtitles: 2


/var/folders/3_/6lgqyskd5v55bylpxy1jbmzr0000gn/T/ipykernel_31313/4144848414.py:15: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  ].str.contains(


,scrape_id,title,subtitle,author,format_raw,publication_date_raw,isbn,release_page
17,18,Relationship Currency,Five Communication Habits For Limitless Influe...,Ravi Rajani,"Paperback, 168 pages","November 4, 2025",9798891389663,2025 Releases
883,884,The Unconventional Entrepreneur,Launch a Successful Business & Live The Workli...,Alexandra Nicole Nolan,"Hardcover, 224 pages","October 4, 2022",9788985071313,2022 Releases


In [118]:
strong_contamination_pattern = (
    r"\bHardcover\b|"
    r"\bPaperback\b|"
    r"\bby\s+.+\($"
)

strong_subtitle_anomalies = (
    leadershipnow_clean.loc[
        leadershipnow_clean[
            "subtitle"
        ].str.contains(
            strong_contamination_pattern,
            case=False,
            na=False,
            regex=True
        ),
        [
            "scrape_id",
            "title",
            "subtitle",
            "author",
            "format_raw",
            "publication_date_raw",
            "isbn",
            "release_page",
            "release_page_url"
        ]
    ]
)

print(
    "Strong subtitle anomalies:",
    len(strong_subtitle_anomalies)
)

strong_subtitle_anomalies

Strong subtitle anomalies: 2


,scrape_id,title,subtitle,author,format_raw,publication_date_raw,isbn,release_page,release_page_url
17,18,Relationship Currency,Five Communication Habits For Limitless Influe...,Ravi Rajani,"Paperback, 168 pages","November 4, 2025",9798891389663,2025 Releases,https://www.leadershipnow.com/leadershop/new20...
883,884,The Unconventional Entrepreneur,Launch a Successful Business & Live The Workli...,Alexandra Nicole Nolan,"Hardcover, 224 pages","October 4, 2022",9788985071313,2022 Releases,https://www.leadershipnow.com/leadershop/new20...


In [119]:
isbn_validity_by_release = (
    leadershipnow_clean
    .groupby(
        "release_page",
        dropna=False
    )
    .agg(
        total_records=(
            "isbn_normalized",
            "size"
        ),
        valid_isbn13=(
            "isbn13_valid",
            lambda x:
                (x == True).sum()
        ),
        invalid_isbn13=(
            "isbn13_valid",
            lambda x:
                (x == False).sum()
        ),
        not_evaluable=(
            "isbn13_valid",
            lambda x:
                x.isna().sum()
        )
    )
    .reset_index()
)

isbn_validity_by_release[
    "invalid_pct"
] = (
    isbn_validity_by_release[
        "invalid_isbn13"
    ]
    /
    isbn_validity_by_release[
        "total_records"
    ]
    * 100
).round(2)

isbn_validity_by_release

,release_page,total_records,valid_isbn13,invalid_isbn13,not_evaluable,invalid_pct
0,2022 Releases,289,281,7,1,2.42
1,2023 Releases,303,277,24,2,7.92
2,2024 Releases,280,252,27,1,9.64
3,2025 Releases,252,251,0,1,0.0


In [120]:
invalid_isbn_audit = (
    leadershipnow_clean.loc[
        leadershipnow_clean[
            "isbn13_valid"
        ] == False,
        [
            "scrape_id",
            "title",
            "author",
            "isbn",
            "isbn_normalized",
            "isbn_prefix",
            "publisher",
            "publication_date_raw",
            "release_page"
        ]
    ]
    .copy()
)

invalid_isbn_audit[
    "first_six_digits"
] = (
    invalid_isbn_audit[
        "isbn_normalized"
    ].str[:6]
)

invalid_isbn_audit[
    "first_six_digits"
].value_counts().head(20)

first_six_digits
978888    21
978898    15
978889     6
978821     2
978125     2
978111     2
978899     1
970889     1
979069     1
979139     1
978835     1
978820     1
978837     1
978165     1
978164     1
978075     1
Name: count, dtype: int64

In [121]:
invalid_isbn_audit[
    "release_page"
].value_counts()

release_page
2024 Releases    27
2023 Releases    24
2022 Releases     7
Name: count, dtype: int64

In [122]:
invalid_isbn_audit[
    "publisher"
].value_counts().head(20)

publisher
ForbesBooks                   8
Maxwell Leadership            5
Amplify Publishing            4
Greenleaf Book Group Press    4
Wiley                         3
Advantage Media Group         2
Post Hill Press               2
Erwin McManus Publishing      1
Parliament Press              1
Princeton University Press    1
BookBaby                      1
Ramsey Press                  1
Amplify Publishing Group      1
Accelerance, Inc.             1
Success Masters LLC           1
Culture Guide                 1
Miles Pond Press              1
Ambika Media                  1
Koehler Books                 1
Scalable Company              1
Name: count, dtype: int64

In [123]:
leadershipnow_clean.loc[
    leadershipnow_clean[
        "isbn_length"
    ] != 13,
    [
        "scrape_id",
        "title",
        "author",
        "isbn",
        "isbn_normalized",
        "isbn_length",
        "publisher",
        "publication_date_raw",
        "release_page",
        "external_book_url"
    ]
]

,scrape_id,title,author,isbn,isbn_normalized,isbn_length,publisher,publication_date_raw,release_page,external_book_url
146,147,The New Emotional Intelligence,Travis Bradberry,979218589660,979218589660,12,Bruyere Publishing,"May 13, 2025",2025 Releases,https://amzn.to/4juExM6
455,456,The Energy of Success,Rebecca Ahmed,97811394245475,97811394245475,14,Wiley,"April 23, 2024",2024 Releases,https://amzn.to/44N2pUO
878,879,When McKinsey Comes to Town,Walt Bogdanich and Michael Forsythe,978385546232,978385546232,12,Doubleday,"October 4, 2022",2022 Releases,https://amzn.to/3QHTdK0


In [124]:
leadershipnow_clean[
    "format_raw_clean"
] = (
    leadershipnow_clean[
        "format_raw"
    ]
    .str.replace(
        "\xa0",
        " ",
        regex=False
    )
    .str.strip()
)

In [125]:
leadershipnow_clean[
    "format"
] = (
    leadershipnow_clean[
        "format_raw_clean"
    ]
    .str.extract(
        r"^\s*([^,]+)",
        expand=False
    )
    .str.strip()
)

In [126]:
leadershipnow_clean[
    "page_count"
] = (
    leadershipnow_clean[
        "format_raw_clean"
    ]
    .str.extract(
        r"(\d+)\s+pages?",
        flags=re.IGNORECASE,
        expand=False
    )
)

In [127]:
leadershipnow_clean[
    "page_count"
] = pd.to_numeric(
    leadershipnow_clean[
        "page_count"
    ],
    errors="coerce"
).astype("Int64")

In [128]:
print(
    "Missing formats:",
    leadershipnow_clean[
        "format"
    ].isna().sum()
)

print(
    "Missing page counts:",
    leadershipnow_clean[
        "page_count"
    ].isna().sum()
)

print(
    "\nFormat distribution:"
)

print(
    leadershipnow_clean[
        "format"
    ].value_counts(
        dropna=False
    )
)

Missing formats: 0
Missing page counts: 0

Format distribution:
format
Hardcover    927
Paperback    196
Hardback       1
Name: count, dtype: int64


In [129]:
leadershipnow_clean[
    "page_count"
].describe()

count        1124.0
mean     278.633452
std       95.946311
min            96.0
25%           224.0
50%           256.0
75%           304.0
max          1200.0
Name: page_count, dtype: Float64

In [130]:
leadershipnow_clean[
    [
        "title",
        "format",
        "page_count",
        "format_raw"
    ]
].sort_values(
    "page_count"
).head(10)

,title,format,page_count,format_raw
367,"Mentally Fit, Powerfully Resilient, Competitiv...",Paperback,96,"Paperback, 96 pages"
274,The Rules of Mentorship,Hardcover,96,"Hardcover, 96 pages"
230,Your Year of Wonders,Paperback,104,"Paperback, 104 pages"
10,Redefining Networking,Hardcover,110,"Hardcover, 110 pages"
416,Difficult Conversations Don't Have to Be Diffi...,Hardcover,112,"Hardcover, 112 pages"
761,Your Time With The Baton,Hardcover,112,"Hardcover, 112 pages"
236,Network Leadership,Hardcover,112,"Hardcover, 112 pages"
263,Everyday Leadership,Paperback,116,"Paperback, 116 pages"
618,Leadership in Star Trek,Paperback,120,"Paperback, 120 pages"
453,Self Less,Hardcover,120,"Hardcover, 120 pages"


In [131]:
leadershipnow_clean[
    [
        "title",
        "format",
        "page_count",
        "format_raw"
    ]
].sort_values(
    "page_count",
    ascending=False
).head(10)

,title,format,page_count,format_raw
141,Mark Twain,Hardcover,1200,"Hardcover, 1200 pages"
559,Areté,Hardcover,1001,"Hardcover, 1001 pages"
332,Reagan,Hardcover,880,"Hardcover, 880 pages"
732,An Ordinary Man,Hardcover,848,"Hardcover, 848 pages"
1081,Watergate,Hardcover,832,"Hardcover, 832 pages"
697,The Corporation and the Twentieth Century,Hardcover,816,"Hardcover, 816 pages"
867,Power Failure,Hardcover,816,"Hardcover, 816 pages"
402,John Quincy Adams,Hardcover,784,"Hardcover, 784 pages"
871,G-Man,Hardcover,752,"Hardcover, 752 pages"
343,Over Work,Hardcover,732,"Hardcover, 732 pages"


In [132]:
leadershipnow_clean[
    "publication_date"
] = pd.to_datetime(
    leadershipnow_clean[
        "publication_date_raw"
    ],
    errors="coerce"
)

In [133]:
print(
    "Successfully parsed dates:",
    leadershipnow_clean[
        "publication_date"
    ].notna().sum()
)

print(
    "Unparsed dates:",
    leadershipnow_clean[
        "publication_date"
    ].isna().sum()
)

Successfully parsed dates: 1120
Unparsed dates: 4


In [134]:
leadershipnow_clean.loc[
    leadershipnow_clean[
        "publication_date"
    ].isna(),
    [
        "title",
        "publication_date_raw",
        "release_page"
    ]
]

,title,publication_date_raw,release_page
538,The Career Workbook,"Workbook edition December 5, 2023",2023 Releases
619,The Conviction to Lead,"September 19, 2023 Revised and Updated edition",2023 Releases
620,Becoming Coachable,"September 19, 2023 Revised and Updated edition",2023 Releases
717,Why Motivating People Doesn't Work...and What ...,"Second Edition May 16, 2023",2023 Releases


In [135]:
leadershipnow_clean[
    "publication_year"
] = (
    leadershipnow_clean[
        "publication_date"
    ]
    .dt.year
    .astype("Int64")
)

In [136]:
leadershipnow_clean[
    "publication_year"
].value_counts().sort_index()

publication_year
2022    289
2023    299
2024    280
2025    252
Name: count, dtype: Int64

In [137]:
leadershipnow_clean[
    "release_page_year"
] = (
    leadershipnow_clean[
        "release_page"
    ]
    .str.extract(
        r"(\d{4})",
        expand=False
    )
    .astype("Int64")
)

In [138]:
leadershipnow_clean[
    "publication_release_year_match"
] = (
    leadershipnow_clean[
        "publication_year"
    ]
    ==
    leadershipnow_clean[
        "release_page_year"
    ]
)

In [139]:
leadershipnow_clean[
    "publication_release_year_match"
].value_counts(
    dropna=False
)

publication_release_year_match
True    1120
<NA>       4
Name: count, dtype: Int64

In [140]:
publication_year_mismatches = (
    leadershipnow_clean.loc[
        leadershipnow_clean[
            "publication_release_year_match"
        ] == False,
        [
            "title",
            "author",
            "publication_date_raw",
            "publication_year",
            "release_page",
            "release_page_year",
            "isbn_normalized"
        ]
    ]
)

print(
    "Publication/release-year mismatches:",
    len(publication_year_mismatches)
)

publication_year_mismatches

Publication/release-year mismatches: 0


,title,author,publication_date_raw,publication_year,release_page,release_page_year,isbn_normalized


In [141]:
date_pattern = (
    r"("
    r"January|February|March|April|May|June|"
    r"July|August|September|October|November|December"
    r")"
    r"\s+\d{1,2},\s+\d{4}"
)

leadershipnow_clean[
    "publication_date_extracted"
] = (
    leadershipnow_clean[
        "publication_date_raw"
    ]
    .str.extract(
        f"({date_pattern})",
        expand=False
    )
    .iloc[:, 0]
)

In [142]:
leadershipnow_clean.loc[
    leadershipnow_clean[
        "publication_date"
    ].isna(),
    [
        "title",
        "publication_date_raw",
        "publication_date_extracted"
    ]
]

,title,publication_date_raw,publication_date_extracted
538,The Career Workbook,"Workbook edition December 5, 2023","December 5, 2023"
619,The Conviction to Lead,"September 19, 2023 Revised and Updated edition","September 19, 2023"
620,Becoming Coachable,"September 19, 2023 Revised and Updated edition","September 19, 2023"
717,Why Motivating People Doesn't Work...and What ...,"Second Edition May 16, 2023","May 16, 2023"


In [143]:
missing_date_mask = (
    leadershipnow_clean[
        "publication_date"
    ].isna()
)

leadershipnow_clean.loc[
    missing_date_mask,
    "publication_date"
] = pd.to_datetime(
    leadershipnow_clean.loc[
        missing_date_mask,
        "publication_date_extracted"
    ],
    errors="coerce"
)

In [144]:
leadershipnow_clean[
    "publication_year"
] = (
    leadershipnow_clean[
        "publication_date"
    ]
    .dt.year
    .astype("Int64")
)

In [145]:
print(
    "Parsed dates:",
    leadershipnow_clean[
        "publication_date"
    ].notna().sum()
)

print(
    "Unparsed dates:",
    leadershipnow_clean[
        "publication_date"
    ].isna().sum()
)

print(
    "\nPublication years:"
)

print(
    leadershipnow_clean[
        "publication_year"
    ]
    .value_counts()
    .sort_index()
)

Parsed dates: 1124
Unparsed dates: 0

Publication years:
publication_year
2022    289
2023    303
2024    280
2025    252
Name: count, dtype: Int64


In [146]:
leadershipnow_clean[
    "publication_release_year_match"
] = (
    leadershipnow_clean[
        "publication_year"
    ]
    ==
    leadershipnow_clean[
        "release_page_year"
    ]
)

leadershipnow_clean[
    "publication_release_year_match"
].value_counts(
    dropna=False
)

publication_release_year_match
True    1124
Name: count, dtype: Int64

In [147]:
def extract_edition_note(value):

    if pd.isna(value):
        return np.nan

    value = str(value)

    date_match = re.search(
        r"(January|February|March|April|May|June|"
        r"July|August|September|October|November|December)"
        r"\s+\d{1,2},\s+\d{4}",
        value,
        flags=re.IGNORECASE
    )

    if not date_match:
        return np.nan

    before = value[
        :date_match.start()
    ].strip()

    after = value[
        date_match.end():
    ].strip()

    note = " ".join(
        part
        for part in [
            before,
            after
        ]
        if part
    )

    return note if note else np.nan

In [148]:
leadershipnow_clean[
    "edition_note"
] = (
    leadershipnow_clean[
        "publication_date_raw"
    ].apply(
        extract_edition_note
    )
)

In [149]:
leadershipnow_clean.loc[
    leadershipnow_clean[
        "edition_note"
    ].notna(),
    [
        "title",
        "publication_date_raw",
        "publication_date",
        "edition_note"
    ]
]

,title,publication_date_raw,publication_date,edition_note
538,The Career Workbook,"Workbook edition December 5, 2023",2023-12-05,Workbook edition
619,The Conviction to Lead,"September 19, 2023 Revised and Updated edition",2023-09-19,Revised and Updated edition
620,Becoming Coachable,"September 19, 2023 Revised and Updated edition",2023-09-19,Revised and Updated edition
717,Why Motivating People Doesn't Work...and What ...,"Second Edition May 16, 2023",2023-05-16,Second Edition


In [150]:
leadershipnow_clean[
    "format"
] = (
    leadershipnow_clean[
        "format"
    ].replace({
        "Hardback": "Hardcover"
    })
)

In [151]:
leadershipnow_clean[
    "format"
].value_counts()

format
Hardcover    928
Paperback    196
Name: count, dtype: int64

In [152]:
leadershipnow_clean[
    "isbn_13"
] = (
    leadershipnow_clean[
        "isbn_normalized"
    ].where(
        leadershipnow_clean[
            "isbn13_valid"
        ] == True
    )
)

In [153]:
print(
    "Source ISBN values:",
    leadershipnow_clean[
        "isbn"
    ].notna().sum()
)

print(
    "Validated ISBN-13:",
    leadershipnow_clean[
        "isbn_13"
    ].notna().sum()
)

print(
    "Not validated as ISBN-13:",
    leadershipnow_clean[
        "isbn_13"
    ].isna().sum()
)

Source ISBN values: 1124
Validated ISBN-13: 1061
Not validated as ISBN-13: 63


In [154]:
def classify_isbn_status(row):

    if row["isbn_length"] != 13:
        return "Invalid length"

    if row["isbn_prefix"] not in [
        "978",
        "979"
    ]:
        return "Invalid prefix"

    if row["isbn13_valid"] == False:
        return "Invalid check digit"

    if row["isbn13_valid"] == True:
        return "Valid ISBN-13"

    return "Not evaluable"

In [156]:
def classify_isbn_status(row):

    # ISBN is not 13 characters long
    if row["isbn_length"] != 13:
        return "Invalid length"

    # ISBN-13 must normally begin with 978 or 979
    if row["isbn_prefix"] not in [
        "978",
        "979"
    ]:
        return "Invalid prefix"

    # Handle missing validation result safely
    if pd.isna(
        row["isbn13_valid"]
    ):
        return "Not evaluable"

    # Check-digit validation
    if row["isbn13_valid"] is False:
        return "Invalid check digit"

    if row["isbn13_valid"] is True:
        return "Valid ISBN-13"

    return "Not evaluable"

In [157]:
def classify_isbn_status(row):

    if row["isbn_length"] != 13:
        return "Invalid length"

    if row["isbn_prefix"] not in [
        "978",
        "979"
    ]:
        return "Invalid prefix"

    validation = row[
        "isbn13_valid"
    ]

    if pd.isna(validation):
        return "Not evaluable"

    if validation == False:
        return "Invalid check digit"

    if validation == True:
        return "Valid ISBN-13"

    return "Not evaluable"

In [158]:
leadershipnow_clean[
    "isbn_status"
] = (
    leadershipnow_clean.apply(
        classify_isbn_status,
        axis=1
    )
)

In [159]:
leadershipnow_clean[
    "isbn_status"
].value_counts(
    dropna=False
)

isbn_status
Valid ISBN-13          1061
Invalid check digit      57
Invalid length            3
Not evaluable             2
Invalid prefix            1
Name: count, dtype: int64

## LeadershipNow ISBN Validation

ISBN values were standardized by removing formatting characters and converting letters to uppercase.

ISBN-13 validation was performed using:

1. identifier length;
2. ISBN prefix;
3. ISBN-13 check-digit calculation.

The audit identified both malformed-length identifiers and 13-digit identifiers that failed check-digit validation.

Invalid source ISBN values were not automatically corrected because their intended values cannot be determined reliably from the collected dataset alone.

The original source identifier is therefore retained in `isbn`, while `isbn_13` contains only identifiers that passed ISBN-13 validation.

This distinction allows questionable source metadata to remain auditable while preventing invalid identifiers from being used as authoritative keys during dataset integration.

In [160]:
source_isbn_frequency = (
    leadershipnow_clean[
        "isbn_normalized"
    ].value_counts()
)

duplicate_source_isbns = (
    source_isbn_frequency[
        source_isbn_frequency > 1
    ].index
)

print(
    "Duplicate ISBN groups:",
    len(
        duplicate_source_isbns
    )
)

Duplicate ISBN groups: 4


In [161]:
duplicate_isbn_records = (
    leadershipnow_clean.loc[
        leadershipnow_clean[
            "isbn_normalized"
        ].isin(
            duplicate_source_isbns
        ),
        [
            "scrape_id",
            "title",
            "subtitle",
            "author",
            "isbn_normalized",
            "isbn13_valid",
            "isbn_status",
            "publisher",
            "publication_date",
            "publication_year",
            "release_page",
            "release_page_year",
            "format",
            "page_count"
        ]
    ]
    .sort_values(
        [
            "isbn_normalized",
            "release_page_year"
        ]
    )
)

duplicate_isbn_records

,scrape_id,title,subtitle,author,isbn_normalized,isbn13_valid,isbn_status,publisher,publication_date,publication_year,release_page,release_page_year,format,page_count
1101,1102,Get It Done,Surprising Lessons from the Science of Motivation,Ayelet Fishbach,9780316538343,True,Valid ISBN-13,"Little, Brown Spark",2022-01-04,2022,2022 Releases,2022,Hardcover,304
813,814,Get It Done,Surprising Lessons from the Science of Motivation,Ayelet Fishbach,9780316538343,True,Valid ISBN-13,"Little, Brown Spark",2023-01-04,2023,2023 Releases,2023,Hardcover,304
1009,1010,Burn Rate,Launching a Startup and Losing My Mind,Andy Dunn,9780593238264,True,Valid ISBN-13,Currency,2022-05-10,2022,2022 Releases,2022,Hardcover,304
713,714,Burn Rate,Launching a Startup and Losing My Mind,Andy Dunn,9780593238264,True,Valid ISBN-13,Currency,2023-05-10,2023,2023 Releases,2023,Hardcover,304
864,865,Win Every Argument,"The Art of Debating, Persuading, and Public Sp...",Mehdi Hasan,9781250853479,True,Valid ISBN-13,Henry Holt and Co.,2022-11-15,2022,2022 Releases,2022,Hardcover,240
809,810,Win Every Argument,"The Art of Debating, Persuading, and Public Sp...",Mehdi Hasan,9781250853479,True,Valid ISBN-13,Henry Holt and Co.,2023-02-28,2023,2023 Releases,2023,Hardcover,336
463,464,You're the Boss,Become the Manager You Want to Be (and Others ...,Sabina Nawaz,9781668023181,True,Valid ISBN-13,Simon & Schuster,2024-03-04,2024,2024 Releases,2024,Hardcover,272
181,182,You're the Boss,Become the Manager You Want to Be (and Others ...,Sabina Nawaz,9781668023181,True,Valid ISBN-13,Simon & Schuster,2025-03-04,2025,2025 Releases,2025,Hardcover,272


In [162]:
duplicate_consistency_columns = [
    "title",
    "author",
    "publisher",
    "format",
    "page_count"
]

duplicate_isbn_consistency = []

for isbn_value, group in (
    duplicate_isbn_records.groupby(
        "isbn_normalized"
    )
):

    result = {
        "isbn_normalized":
            isbn_value,

        "records":
            len(group)
    }

    for column in (
        duplicate_consistency_columns
    ):

        result[
            f"{column}_unique"
        ] = (
            group[column]
            .nunique(
                dropna=False
            )
        )

    duplicate_isbn_consistency.append(
        result
    )

duplicate_isbn_consistency_df = (
    pd.DataFrame(
        duplicate_isbn_consistency
    )
)

duplicate_isbn_consistency_df

,isbn_normalized,records,title_unique,author_unique,publisher_unique,format_unique,page_count_unique
0,9780316538343,2,1,1,1,1,1
1,9780593238264,2,1,1,1,1,1
2,9781250853479,2,1,1,1,1,2
3,9781668023181,2,1,1,1,1,1


In [163]:
duplicate_date_audit = (
    duplicate_isbn_records[
        [
            "isbn_normalized",
            "title",
            "publication_date",
            "publication_year",
            "release_page_year"
        ]
    ]
)

duplicate_date_audit

,isbn_normalized,title,publication_date,publication_year,release_page_year
1101,9780316538343,Get It Done,2022-01-04,2022,2022
813,9780316538343,Get It Done,2023-01-04,2023,2023
1009,9780593238264,Burn Rate,2022-05-10,2022,2022
713,9780593238264,Burn Rate,2023-05-10,2023,2023
864,9781250853479,Win Every Argument,2022-11-15,2022,2022
809,9781250853479,Win Every Argument,2023-02-28,2023,2023
463,9781668023181,You're the Boss,2024-03-04,2024,2024
181,9781668023181,You're the Boss,2025-03-04,2025,2025


In [164]:
title_frequency = (
    leadershipnow_clean[
        "title"
    ].value_counts()
)

duplicate_titles = (
    title_frequency[
        title_frequency > 1
    ]
)

print(
    "Titles appearing more than once:",
    len(duplicate_titles)
)

duplicate_titles

Titles appearing more than once: 10


title
The Power to Change        2
Seeing Around Corners      2
You're the Boss            2
A New Kind of Diversity    2
Positive Influence         2
Dancing with Disruption    2
Burn Rate                  2
Win Every Argument         2
Get It Done                2
Talent                     2
Name: count, dtype: int64

In [165]:
duplicate_title_records = (
    leadershipnow_clean.loc[
        leadershipnow_clean[
            "title"
        ].isin(
            duplicate_titles.index
        ),
        [
            "title",
            "subtitle",
            "author",
            "isbn_normalized",
            "isbn13_valid",
            "publisher",
            "publication_date",
            "format",
            "page_count",
            "release_page"
        ]
    ]
    .sort_values(
        [
            "title",
            "publication_date"
        ]
    )
)

duplicate_title_records

,title,subtitle,author,isbn_normalized,isbn13_valid,publisher,publication_date,format,page_count,release_page
904,A New Kind of Diversity,Making the Different Generations on Your Team ...,Tim Elmore,9788887100005,False,Maxwell Leadership,2022-10-25,Hardcover,304,2022 Releases
593,A New Kind of Diversity,Making the Different Generations on Your Team ...,Tim Elmore / Foreward by John C. Maxwell,9798887100005,True,Maxwell Leadership,2023-10-25,Hardcover,304,2023 Releases
1009,Burn Rate,Launching a Startup and Losing My Mind,Andy Dunn,9780593238264,True,Currency,2022-05-10,Hardcover,304,2022 Releases
713,Burn Rate,Launching a Startup and Losing My Mind,Andy Dunn,9780593238264,True,Currency,2023-05-10,Hardcover,304,2023 Releases
776,Dancing with Disruption,Leading Dramatic Change During Global Transfor...,Jeff Skipper,9781738903801,True,Peacebridge Publishing,2023-03-22,Paperback,162,2023 Releases
709,Dancing with Disruption,A New Approach to Navigating Lifes Biggest Ch...,Linda Rossetti,9781538169377,True,Rowman & Littlefield Publishers,2023-05-05,Hardcover,196,2023 Releases
1101,Get It Done,Surprising Lessons from the Science of Motivation,Ayelet Fishbach,9780316538343,True,"Little, Brown Spark",2022-01-04,Hardcover,304,2022 Releases
813,Get It Done,Surprising Lessons from the Science of Motivation,Ayelet Fishbach,9780316538343,True,"Little, Brown Spark",2023-01-04,Hardcover,304,2023 Releases
729,Positive Influence,NaN,Brian Smith and Mary Griffin,9781641467629,True,Made for Success,2023-04-04,Hardcover,350,2023 Releases
701,Positive Influence,The First and Last Mile of Leadership,Tsun-yan Hsieh and Huijin Kong,9781944660567,True,World Scientific Publishing Company,2023-06-28,Hardcover,360,2023 Releases


## Duplicate Record Resolution

Duplicate analysis identified four ISBN groups in which the same validated ISBN-13 appeared on two LeadershipNow release archive pages.

Within each group, title, author, publisher, and format were consistent. The duplicate records therefore represent repeated archive appearances of the same underlying book rather than separate books.

Publication dates differed between archive appearances, and one duplicate group also contained a page-count difference. Therefore, duplicate resolution is not performed using an arbitrary first-record or last-record rule.

A canonical record will be selected using the most internally consistent bibliographic record, while all source archive appearances will be retained in provenance fields for auditability.

Books sharing the same title but having different authors or ISBNs are treated as distinct works and are not deduplicated by title.

In [166]:
isbn_provenance = (
    leadershipnow_clean
    .groupby(
        "isbn_normalized",
        dropna=False
    )
    .agg(
        archive_appearances=(
            "scrape_id",
            "size"
        ),
        release_pages=(
            "release_page",
            lambda x:
                list(dict.fromkeys(x))
        ),
        release_page_years=(
            "release_page_year",
            lambda x:
                list(dict.fromkeys(x))
        ),
        source_publication_dates=(
            "publication_date",
            lambda x:
                list(dict.fromkeys(x))
        )
    )
    .reset_index()
)

In [167]:
isbn_provenance.loc[
    isbn_provenance[
        "archive_appearances"
    ] > 1
]

,isbn_normalized,archive_appearances,release_pages,release_page_years,source_publication_dates
111,9780316538343,2,"[2023 Releases, 2022 Releases]","[2023, 2022]","[2023-01-04 00:00:00, 2022-01-04 00:00:00]"
147,9780593238264,2,"[2023 Releases, 2022 Releases]","[2023, 2022]","[2023-05-10 00:00:00, 2022-05-10 00:00:00]"
335,9781250853479,2,"[2023 Releases, 2022 Releases]","[2023, 2022]","[2023-02-28 00:00:00, 2022-11-15 00:00:00]"
853,9781668023181,2,"[2025 Releases, 2024 Releases]","[2025, 2024]","[2025-03-04 00:00:00, 2024-03-04 00:00:00]"


In [168]:
leadershipnow_clean[
    "duplicate_isbn_group"
] = (
    leadershipnow_clean[
        "isbn_normalized"
    ].duplicated(
        keep=False
    )
)

In [169]:
leadershipnow_clean[
    "duplicate_isbn_group"
].value_counts()

duplicate_isbn_group
False    1116
True        8
Name: count, dtype: int64

In [170]:
leadershipnow_clean[
    "canonical_record"
] = True

In [171]:
duplicate_review = (
    duplicate_isbn_records[
        [
            "scrape_id",
            "title",
            "author",
            "isbn_normalized",
            "publication_date",
            "page_count",
            "release_page"
        ]
    ]
    .copy()
)

duplicate_review[
    "keep_canonical"
] = pd.NA

duplicate_review

,scrape_id,title,author,isbn_normalized,publication_date,page_count,release_page,keep_canonical
1101,1102,Get It Done,Ayelet Fishbach,9780316538343,2022-01-04,304,2022 Releases,<NA>
813,814,Get It Done,Ayelet Fishbach,9780316538343,2023-01-04,304,2023 Releases,<NA>
1009,1010,Burn Rate,Andy Dunn,9780593238264,2022-05-10,304,2022 Releases,<NA>
713,714,Burn Rate,Andy Dunn,9780593238264,2023-05-10,304,2023 Releases,<NA>
864,865,Win Every Argument,Mehdi Hasan,9781250853479,2022-11-15,240,2022 Releases,<NA>
809,810,Win Every Argument,Mehdi Hasan,9781250853479,2023-02-28,336,2023 Releases,<NA>
463,464,You're the Boss,Sabina Nawaz,9781668023181,2024-03-04,272,2024 Releases,<NA>
181,182,You're the Boss,Sabina Nawaz,9781668023181,2025-03-04,272,2025 Releases,<NA>


In [172]:
leadershipnow_clean[
    "duplicate_title"
] = (
    leadershipnow_clean[
        "title"
    ].duplicated(
        keep=False
    )
)

In [173]:
pd.crosstab(
    leadershipnow_clean[
        "duplicate_title"
    ],
    leadershipnow_clean[
        "duplicate_isbn_group"
    ]
)

duplicate_isbn_group,False,True
duplicate_title,,
False,1104,0
True,12,8


In [174]:
leadershipnow_clean.loc[
    leadershipnow_clean[
        "isbn_status"
    ] == "Not evaluable",
    [
        "scrape_id",
        "title",
        "author",
        "isbn",
        "isbn_normalized",
        "isbn_length",
        "isbn_prefix",
        "isbn13_valid"
    ]
]

,scrape_id,title,author,isbn,isbn_normalized,isbn_length,isbn_prefix,isbn13_valid
810,811,Disruptable,Allan Young,978194663316X,978194663316X,13,978,<NA>
819,820,Buy Back Your Time,Dan Martell,978059342297X,978059342297X,13,978,<NA>


In [175]:
strong_contamination_pattern = (
    r"\bHardcover\b|"
    r"\bPaperback\b|"
    r"\bby\s+.+\($"
)

strong_subtitle_anomalies = (
    leadershipnow_clean.loc[
        leadershipnow_clean[
            "subtitle"
        ].str.contains(
            strong_contamination_pattern,
            case=False,
            na=False,
            regex=True
        ),
        [
            "scrape_id",
            "title",
            "subtitle",
            "author",
            "format_raw",
            "publication_date_raw",
            "isbn_normalized",
            "release_page"
        ]
    ]
)

print(
    "Strong subtitle anomalies:",
    len(
        strong_subtitle_anomalies
    )
)

strong_subtitle_anomalies

Strong subtitle anomalies: 2


,scrape_id,title,subtitle,author,format_raw,publication_date_raw,isbn_normalized,release_page
17,18,Relationship Currency,Five Communication Habits For Limitless Influe...,Ravi Rajani,"Paperback, 168 pages","November 4, 2025",9798891389663,2025 Releases
883,884,The Unconventional Entrepreneur,Launch a Successful Business & Live The Workli...,Alexandra Nicole Nolan,"Hardcover, 224 pages","October 4, 2022",9788985071313,2022 Releases


In [176]:
text_columns = [
    "title",
    "subtitle",
    "author",
    "publisher"
]

encoding_audit = []

for column in text_columns:

    affected = (
        leadershipnow_clean[
            column
        ]
        .astype("string")
        .str.contains(
            "",
            regex=False,
            na=False
        )
    )

    encoding_audit.append({
        "column": column,
        "affected_records":
            affected.sum()
    })

pd.DataFrame(
    encoding_audit
)

,column,affected_records
0,title,4
1,subtitle,16
2,author,1
3,publisher,0


In [177]:
encoding_mask = False

for column in text_columns:

    encoding_mask = (
        encoding_mask
        |
        leadershipnow_clean[
            column
        ]
        .astype("string")
        .str.contains(
            "",
            regex=False,
            na=False
        )
    )

leadershipnow_clean.loc[
    encoding_mask,
    [
        "title",
        "subtitle",
        "author",
        "publisher"
    ]
]

,title,subtitle,author,publisher
115,Shoveling $h!t,A Love Story About the Entrepreneurs Messy Pa...,Kass Lazerow and Michael Lazerow,Amplify Publishing
235,Get Off the X,CIA Secrets for Conquering Obstacles and Achie...,Michele Rigby Assad,Dexterity
281,More Than Pretty Boxes,How the Rise of Professional Organizing Shows ...,Carrie M. Lane,University of Chicago Press
348,AI Snake Oil,"What Artificial Intelligence Can Do, What It C...",Arvind Narayanan and Sayash Kapoor,Princeton University Press
446,The Generous Leader,7 Ways to Give of Yourself for Everyones Gain,Joe Davis,Berrett-Koehler Publishers
481,Conquer Your Culture,"CEOs Simple, Proven Guide to an Exceptional a...",David Komar,Culture Guide
511,The Farmers Code,How Legacies are Built,Mike C. Young,ForbesBooks
512,Leading for Impact,The CEOs Guide to Influencing with Integrity,Jennifer Schielke,Advantage Media Group
534,Poor Charlies Almanack,The Essential Wit and Wisdom of Charles T. Munger,Charles T. Munger,Stripe Press
536,Moonshot,A NASA Astronauts Guide to Achieving the Impo...,Mike Massimino,Hachette Go


In [178]:
quality_audit = pd.DataFrame({
    "metric": [
        "Total records",
        "Missing titles",
        "Missing authors",
        "Missing publishers",
        "Missing publication dates",
        "Missing publication years",
        "Missing formats",
        "Missing page counts",
        "Validated ISBN-13",
        "Invalid/unvalidated ISBN",
        "Duplicate ISBN records",
        "Duplicate ISBN groups",
        "Duplicate-title records"
    ],

    "value": [
        len(
            leadershipnow_clean
        ),

        leadershipnow_clean[
            "title"
        ].isna().sum(),

        leadershipnow_clean[
            "author"
        ].isna().sum(),

        leadershipnow_clean[
            "publisher"
        ].isna().sum(),

        leadershipnow_clean[
            "publication_date"
        ].isna().sum(),

        leadershipnow_clean[
            "publication_year"
        ].isna().sum(),

        leadershipnow_clean[
            "format"
        ].isna().sum(),

        leadershipnow_clean[
            "page_count"
        ].isna().sum(),

        leadershipnow_clean[
            "isbn_13"
        ].notna().sum(),

        leadershipnow_clean[
            "isbn_13"
        ].isna().sum(),

        leadershipnow_clean[
            "duplicate_isbn_group"
        ].sum(),

        len(
            duplicate_source_isbns
        ),

        leadershipnow_clean[
            "duplicate_title"
        ].sum()
    ]
})

quality_audit

,metric,value
0,Total records,1124
1,Missing titles,0
2,Missing authors,0
3,Missing publishers,0
4,Missing publication dates,0
5,Missing publication years,0
6,Missing formats,0
7,Missing page counts,0
8,Validated ISBN-13,1061
9,Invalid/unvalidated ISBN,63


In [179]:
def classify_isbn_status(row):

    isbn = row["isbn_normalized"]

    if pd.isna(isbn):
        return "Missing ISBN"

    isbn = str(isbn)

    if len(isbn) != 13:
        return "Invalid length"

    if not isbn.isdigit():
        return "Invalid characters"

    if isbn[:3] not in [
        "978",
        "979"
    ]:
        return "Invalid prefix"

    validation = row[
        "isbn13_valid"
    ]

    if pd.isna(validation):
        return "Not evaluable"

    if validation == False:
        return "Invalid check digit"

    if validation == True:
        return "Valid ISBN-13"

    return "Not evaluable"

In [180]:
leadershipnow_clean[
    "isbn_status"
] = (
    leadershipnow_clean.apply(
        classify_isbn_status,
        axis=1
    )
)

leadershipnow_clean[
    "isbn_status"
].value_counts(
    dropna=False
)

isbn_status
Valid ISBN-13          1061
Invalid check digit      57
Invalid length            3
Invalid characters        2
Invalid prefix            1
Name: count, dtype: int64

In [181]:
for column in [
    "title",
    "subtitle",
    "author",
    "publisher"
]:

    leadershipnow_clean[
        column
    ] = (
        leadershipnow_clean[
            column
        ]
        .astype("string")
        .str.replace(
            "",
            "’",
            regex=False
        )
    )

In [182]:
for column in [
    "title",
    "subtitle",
    "author",
    "publisher"
]:

    remaining = (
        leadershipnow_clean[
            column
        ]
        .str.contains(
            "",
            regex=False,
            na=False
        )
        .sum()
    )

    print(
        column,
        remaining
    )

title 0
subtitle 0
author 0
publisher 0


In [183]:
for _, row in (
    strong_subtitle_anomalies.iterrows()
):

    print(
        "\nTITLE:",
        row["title"]
    )

    print(
        "SUBTITLE:",
        repr(
            row["subtitle"]
        )
    )

    print(
        "AUTHOR:",
        row["author"]
    )

    print(
        "FORMAT:",
        row["format_raw"]
    )

    print(
        "DATE:",
        row["publication_date_raw"]
    )


TITLE: Relationship Currency
SUBTITLE: 'Five Communication Habits For Limitless Influence and Business Success Hardcover \x96 November 4, 2025 by Ravi Rajani ('
AUTHOR: Ravi Rajani
FORMAT: Paperback, 168 pages
DATE: November 4, 2025

TITLE: The Unconventional Entrepreneur
SUBTITLE: 'Launch a Successful Business & Live The Worklife Dream Hardcover'
AUTHOR: Alexandra Nicole Nolan
FORMAT: Hardcover, 224 pages
DATE: October 4, 2022


In [184]:
temporary_columns = [
    "isbn_length",
    "isbn_prefix",
    "format_raw_clean",
    "publication_date_extracted",
    "release_page_year",
    "publication_release_year_match",
    "canonical_record"
]

leadershipnow_clean = (
    leadershipnow_clean.drop(
        columns=temporary_columns,
        errors="ignore"
    )
)

In [185]:
openlibrary_work_clean = (
    openlibrary_work_clean.drop(
        columns=[
            "description_length",
            "work_subject_count"
        ],
        errors="ignore"
    )
)

In [186]:
print(
    "Rows:",
    len(
        leadershipnow_clean
    )
)

print(
    "Columns:",
    leadershipnow_clean.shape[1]
)

print(
    "Missing titles:",
    leadershipnow_clean[
        "title"
    ].isna().sum()
)

print(
    "Missing authors:",
    leadershipnow_clean[
        "author"
    ].isna().sum()
)

print(
    "Missing publishers:",
    leadershipnow_clean[
        "publisher"
    ].isna().sum()
)

print(
    "Missing publication dates:",
    leadershipnow_clean[
        "publication_date"
    ].isna().sum()
)

print(
    "Missing page counts:",
    leadershipnow_clean[
        "page_count"
    ].isna().sum()
)

print(
    "Validated ISBN-13:",
    leadershipnow_clean[
        "isbn_13"
    ].notna().sum()
)

print(
    "Source duplicate ISBN records:",
    leadershipnow_clean[
        "duplicate_isbn_group"
    ].sum()
)

Rows: 1124
Columns: 26
Missing titles: 0
Missing authors: 0
Missing publishers: 0
Missing publication dates: 0
Missing page counts: 0
Validated ISBN-13: 1061
Source duplicate ISBN records: 8


## LeadershipNow Cleaning Interpretation

The LeadershipNow dataset retained all 1,124 source records collected from the complete 2022–2025 release archives.

Bibliographic text fields were normalized conservatively while preserving their original capitalization and semantic content. Encoding artifacts affecting apostrophes were corrected after inspection.

Book format and page count were successfully extracted for all records. Publication dates were standardized for all 1,124 records, including four records whose raw date strings contained edition information. Edition information was preserved separately rather than discarded.

ISBN values underwent structural and ISBN-13 check-digit validation. Validated ISBN-13 values are stored separately from the original source identifiers so that questionable source metadata remains auditable without being treated as an authoritative matching key.

Duplicate analysis identified four ISBN groups representing eight source records appearing across multiple release archives. These records are intentionally retained at the source-cleaning stage because archive appearance constitutes source provenance. Canonical book-level deduplication will be performed during data integration.

Duplicate titles were not treated as duplicates automatically because multiple distinct books legitimately share the same title.

The resulting cleaned dataset therefore represents a standardized but provenance-preserving version of the LeadershipNow source.

## Correction of Contaminated Subtitle Records

Inspection identified two subtitle records containing bibliographic metadata that had been captured as part of the subtitle during source extraction.

Because only two records were affected and their legitimate subtitle boundaries were clearly identifiable, targeted corrections were applied rather than a broad automated rule that could alter valid subtitle text.

The corrections affect only the cleaned dataset. The original raw dataset remains unchanged and preserves the source extraction exactly as collected.

In [187]:
# Correct the two confirmed contaminated subtitles

subtitle_corrections = {
    "Relationship Currency":
        "Five Communication Habits For Limitless Influence and Business Success",

    "The Unconventional Entrepreneur":
        "Launch a Successful Business & Live The Worklife Dream"
}

for title, corrected_subtitle in subtitle_corrections.items():

    leadershipnow_clean.loc[
        leadershipnow_clean["title"] == title,
        "subtitle"
    ] = corrected_subtitle

In [188]:
leadershipnow_clean.loc[
    leadershipnow_clean["title"].isin(
        subtitle_corrections.keys()
    ),
    [
        "title",
        "subtitle",
        "author",
        "format",
        "publication_date"
    ]
]

,title,subtitle,author,format,publication_date
17,Relationship Currency,Five Communication Habits For Limitless Influe...,Ravi Rajani,Paperback,2025-11-04
883,The Unconventional Entrepreneur,Launch a Successful Business & Live The Workli...,Alexandra Nicole Nolan,Hardcover,2022-10-04


In [189]:
strong_contamination_pattern = (
    r"\bHardcover\b|"
    r"\bPaperback\b|"
    r"\bby\s+.+\($"
)

remaining_subtitle_anomalies = (
    leadershipnow_clean.loc[
        leadershipnow_clean["subtitle"].str.contains(
            strong_contamination_pattern,
            case=False,
            na=False,
            regex=True
        ),
        [
            "title",
            "subtitle",
            "author"
        ]
    ]
)

print(
    "Remaining strong subtitle anomalies:",
    len(remaining_subtitle_anomalies)
)

remaining_subtitle_anomalies

Remaining strong subtitle anomalies: 0


,title,subtitle,author


In [190]:
for column in [
    "title",
    "subtitle",
    "author",
    "publisher"
]:

    leadershipnow_clean[column] = (
        leadershipnow_clean[column]
        .astype("string")
        .str.replace(
            "",
            "’",
            regex=False
        )
    )

In [191]:
for column in [
    "title",
    "subtitle",
    "author",
    "publisher"
]:

    remaining = (
        leadershipnow_clean[column]
        .str.contains(
            "",
            regex=False,
            na=False
        )
        .sum()
    )

    print(
        f"{column}: {remaining}"
    )

title: 0
subtitle: 0
author: 0
publisher: 0


In [192]:
print("LEADERSHIPNOW FINAL CLEANING AUDIT")
print("=" * 45)

print(
    "Rows:",
    len(leadershipnow_clean)
)

print(
    "Columns:",
    leadershipnow_clean.shape[1]
)

print(
    "Missing titles:",
    leadershipnow_clean["title"].isna().sum()
)

print(
    "Missing subtitles:",
    leadershipnow_clean["subtitle"].isna().sum()
)

print(
    "Missing authors:",
    leadershipnow_clean["author"].isna().sum()
)

print(
    "Missing publishers:",
    leadershipnow_clean["publisher"].isna().sum()
)

print(
    "Missing publication dates:",
    leadershipnow_clean["publication_date"].isna().sum()
)

print(
    "Missing publication years:",
    leadershipnow_clean["publication_year"].isna().sum()
)

print(
    "Missing formats:",
    leadershipnow_clean["format"].isna().sum()
)

print(
    "Missing page counts:",
    leadershipnow_clean["page_count"].isna().sum()
)

print(
    "Validated ISBN-13:",
    leadershipnow_clean["isbn_13"].notna().sum()
)

print(
    "Invalid/unvalidated ISBN:",
    leadershipnow_clean["isbn_13"].isna().sum()
)

print(
    "Source duplicate ISBN records:",
    leadershipnow_clean["duplicate_isbn_group"].sum()
)

print(
    "Duplicate-title records:",
    leadershipnow_clean["duplicate_title"].sum()
)

LEADERSHIPNOW FINAL CLEANING AUDIT
Rows: 1124
Columns: 26
Missing titles: 0
Missing subtitles: 43
Missing authors: 0
Missing publishers: 0
Missing publication dates: 0
Missing publication years: 0
Missing formats: 0
Missing page counts: 0
Validated ISBN-13: 1061
Invalid/unvalidated ISBN: 63
Source duplicate ISBN records: 8
Duplicate-title records: 20


In [193]:
temporary_columns = [
    "isbn_length",
    "isbn_prefix",
    "format_raw_clean",
    "publication_date_extracted",
    "release_page_year",
    "publication_release_year_match",
    "canonical_record"
]

leadershipnow_clean = (
    leadershipnow_clean.drop(
        columns=temporary_columns,
        errors="ignore"
    )
)

In [194]:
openlibrary_work_clean = (
    openlibrary_work_clean.drop(
        columns=[
            "description_length",
            "work_subject_count"
        ],
        errors="ignore"
    )
)

In [195]:
print("OPEN LIBRARY SEARCH")
print("-" * 40)
print("Shape:", openlibrary_search_clean.shape)
print(
    "Unique works:",
    openlibrary_search_clean["openlibrary_key"].nunique()
)
print(
    "Duplicate work keys:",
    openlibrary_search_clean["openlibrary_key"].duplicated().sum()
)

print("\nOPEN LIBRARY WORK ENRICHMENT")
print("-" * 40)
print("Shape:", openlibrary_work_clean.shape)
print(
    "Unique works:",
    openlibrary_work_clean["openlibrary_key"].nunique()
)
print(
    "Duplicate work keys:",
    openlibrary_work_clean["openlibrary_key"].duplicated().sum()
)

print("\nLEADERSHIPNOW")
print("-" * 40)
print("Shape:", leadershipnow_clean.shape)
print(
    "Validated ISBN-13:",
    leadershipnow_clean["isbn_13"].notna().sum()
)
print(
    "Source duplicate ISBN records:",
    leadershipnow_clean["duplicate_isbn_group"].sum()
)

OPEN LIBRARY SEARCH
----------------------------------------
Shape: (1000, 31)
Unique works: 950
Duplicate work keys: 50

OPEN LIBRARY WORK ENRICHMENT
----------------------------------------
Shape: (950, 13)
Unique works: 950
Duplicate work keys: 0

LEADERSHIPNOW
----------------------------------------
Shape: (1124, 26)
Validated ISBN-13: 1061
Source duplicate ISBN records: 8


In [196]:
from pathlib import Path

processed_dir = Path(
    "../data/processed"
)

processed_dir.mkdir(
    parents=True,
    exist_ok=True
)

In [197]:
openlibrary_search_clean.to_csv(
    processed_dir /
    "open_library_search_clean.csv",
    index=False
)

openlibrary_work_clean.to_csv(
    processed_dir /
    "open_library_work_enrichment_clean.csv",
    index=False
)

leadershipnow_clean.to_csv(
    processed_dir /
    "leadershipnow_books_clean.csv",
    index=False
)

print("Processed datasets saved successfully.")

Processed datasets saved successfully.


In [198]:
for file_path in sorted(
    processed_dir.glob("*.csv")
):

    print(
        file_path.name,
        "->",
        round(
            file_path.stat().st_size / 1024,
            2
        ),
        "KB"
    )

leadershipnow_books_clean.csv -> 539.9 KB
open_library_search_clean.csv -> 1011.05 KB
open_library_work_enrichment_clean.csv -> 353.65 KB


## Processed CSV Serialization Note

Several Open Library variables contain list-valued metadata, including authors, publishers, ISBN collections, languages, subjects, and Work API subject fields.

These variables were reconstructed as Python lists during cleaning. When the processed datasets are saved as CSV files, Python lists are serialized as text representations because CSV does not support native list data types.

Therefore, downstream notebooks that reload these CSV files must parse the relevant serialized list columns before performing list-based operations.

This serialization behavior does not represent data loss; it is a limitation of the CSV storage format.

# Conclusion

Notebook 03 completed the cleaning and validation of the three source datasets used in the Leadership and Management Book Recommendation System.

### Open Library Search API

The original 1,000 API collection records were consolidated into 950 unique Open Library works. Repeated work keys were identified as retrieval overlap across the predefined search queries rather than independent book records. All query memberships were preserved.

List-valued metadata was reconstructed from CSV serialization, text fields were normalized conservatively, publication years were validated, and rating information was retained without treating missing ratings as zero.

### Open Library Work API

All 950 Work API records were successfully aligned with the 950 cleaned Search API works using `openlibrary_key`.

Work-level metadata, including subjects, descriptions, classification information, excerpts, and cover identifiers, was cleaned while preserving genuine missingness. Description coverage remained limited, supporting the later use of multiple textual features for NLP rather than description alone.

### LeadershipNow

All 1,124 source records from the complete 2022–2025 release archives were retained.

Format and page-count information was successfully standardized for all records. Publication dates were parsed for all records, including four records containing edition information, with edition notes preserved separately.

ISBN values were normalized and validated structurally and through ISBN-13 check-digit verification. Valid ISBN-13 identifiers were separated from questionable source identifiers so that invalid metadata remains auditable without being used as an authoritative integration key.

Two contaminated subtitle records were corrected after direct inspection, and confirmed character-encoding artifacts were standardized.

Duplicate analysis identified four ISBN groups representing eight repeated source records across release archives. These records were intentionally retained during source cleaning because they preserve archive provenance. Canonical book-level deduplication will be performed during data integration.

### Output

The cleaned datasets are saved as:

- `data/processed/open_library_search_clean.csv`
- `data/processed/open_library_work_enrichment_clean.csv`
- `data/processed/leadershipnow_books_clean.csv`

These datasets provide the validated source layer for integration and cross-source deduplication.

# Next Step

The next stage is **Notebook 04 — Data Integration and Deduplication**.

The integration workflow will:

1. merge Open Library Search and Work API records using `openlibrary_key`;
2. create a unified Open Library book-level dataset;
3. compare validated ISBN identifiers across Open Library and LeadershipNow;
4. resolve repeated LeadershipNow archive records into canonical book records;
5. identify cross-source duplicate books;
6. use conservative title-and-author matching for records without reliable ISBN matches;
7. preserve source provenance and matching methodology;
8. produce the integrated dataset for SQL, EDA, statistical analysis, NLP, clustering, and recommendation modeling.

In [199]:
print(openlibrary_search_clean.columns.tolist())

['openlibrary_key', 'title', 'authors', 'author_keys', 'first_publish_year', 'publish_dates', 'publishers', 'isbn_10', 'isbn_13', 'all_isbns', 'languages', 'subjects', 'edition_count', 'ratings_average', 'ratings_count', 'ratings_count_1', 'ratings_count_2', 'ratings_count_3', 'ratings_count_4', 'ratings_count_5', 'want_to_read_count', 'currently_reading_count', 'already_read_count', 'cover_id', 'cover_url', 'ebook_access', 'has_fulltext', 'public_scan', 'collection_query', 'source', 'source_url']


In [200]:
query_membership = (
    openlibrary_search_clean
    .groupby("openlibrary_key")["collection_query"]
    .agg(
        lambda x: list(
            dict.fromkeys(
                value
                for value in x
                if pd.notna(value)
            )
        )
    )
    .reset_index(
        name="collection_queries"
    )
)

query_membership.head()

,openlibrary_key,collection_queries
0,/works/OL102817W,"[management, project management]"
1,/works/OL10932218W,[executive leadership]
2,/works/OL11019317W,[leadership development]
3,/works/OL11150616W,[executive leadership]
4,/works/OL11274839W,"[people management, human resource management]"


In [201]:
openlibrary_search_work_clean = (
    openlibrary_search_clean
    .drop(
        columns=[
            "collection_query"
        ]
    )
    .drop_duplicates(
        subset=[
            "openlibrary_key"
        ],
        keep="first"
    )
    .merge(
        query_membership,
        on="openlibrary_key",
        how="left"
    )
)

In [202]:
openlibrary_search_work_clean[
    "query_match_count"
] = (
    openlibrary_search_work_clean[
        "collection_queries"
    ].apply(len)
)

In [203]:
print(
    "Original collection rows:",
    len(openlibrary_search_clean)
)

print(
    "Consolidated work rows:",
    len(openlibrary_search_work_clean)
)

print(
    "Unique works:",
    openlibrary_search_work_clean[
        "openlibrary_key"
    ].nunique()
)

print(
    "Duplicate work keys:",
    openlibrary_search_work_clean[
        "openlibrary_key"
    ].duplicated().sum()
)

print(
    "Total query memberships preserved:",
    openlibrary_search_work_clean[
        "query_match_count"
    ].sum()
)

print(
    "Maximum query matches:",
    openlibrary_search_work_clean[
        "query_match_count"
    ].max()
)

Original collection rows: 1000
Consolidated work rows: 950
Unique works: 950
Duplicate work keys: 0
Total query memberships preserved: 1000
Maximum query matches: 6


In [204]:
openlibrary_search_work_clean[
    "query_match_count"
].value_counts().sort_index()

query_match_count
1    908
2     37
3      4
6      1
Name: count, dtype: int64

In [205]:
(
    openlibrary_search_work_clean[
        "query_match_count"
    ]
    .value_counts()
    .sort_index()
)

query_match_count
1    908
2     37
3      4
6      1
Name: count, dtype: int64

In [206]:
openlibrary_search_work_clean[
    [
        "title",
        "authors",
        "collection_queries",
        "query_match_count"
    ]
].sort_values(
    "query_match_count",
    ascending=False
).head(15)

,title,authors,collection_queries,query_match_count
1,Leadership in Organizations,[Gary A. Yukl],"[leadership, leadership development, executive...",6
293,Human resource management,"[R. Wayne Mondy, Robert M. Noe, Shane R. Preme...","[management, performance management, human res...",3
289,Strategic management,"[Gregory G. Dess, G. T. Lumpkin, Alan B. Eisne...","[management, strategic management, innovation ...",3
290,Management,[Stephen P. Robbins],"[management, change management, innovation man...",3
295,Strategic management and business policy,"[Thomas L. Wheelen, J. David Hunger, Tom Wheelen]","[management, strategic management, innovation ...",3
311,Strategic management,"[Pearce, John A., John A. Pearce, Richard B. R...","[management, strategic management]",2
310,Strategic market management,[David A. Aaker],"[management, strategic management]",2
473,Operations management,[Roger G. Schroeder],"[operations management, decision making]",2
36,Improving organizational effectiveness through...,"[Bernard M. Bass, Bruce J. Avolio]","[leadership, transformational leadership]",2
280,Marketing management,[Philip Kotler],"[management, change management]",2


In [207]:
openlibrary_search_clean = (
    openlibrary_search_work_clean.copy()
)

In [208]:
print(
    "Shape:",
    openlibrary_search_clean.shape
)

print(
    "Unique works:",
    openlibrary_search_clean[
        "openlibrary_key"
    ].nunique()
)

print(
    "Duplicate work keys:",
    openlibrary_search_clean[
        "openlibrary_key"
    ].duplicated().sum()
)

print(
    "Query memberships:",
    openlibrary_search_clean[
        "query_match_count"
    ].sum()
)

Shape: (950, 32)
Unique works: 950
Duplicate work keys: 0
Query memberships: 1000


In [209]:
openlibrary_search_clean.to_csv(
    processed_dir /
    "open_library_search_clean.csv",
    index=False
)

print(
    "Corrected Open Library Search file saved."
)

Corrected Open Library Search file saved.


In [210]:
verification_df = pd.read_csv(
    processed_dir /
    "open_library_search_clean.csv"
)

print(
    "Saved rows:",
    len(verification_df)
)

print(
    "Saved unique works:",
    verification_df[
        "openlibrary_key"
    ].nunique()
)

print(
    "Saved duplicate work keys:",
    verification_df[
        "openlibrary_key"
    ].duplicated().sum()
)

print(
    "File size:",
    round(
        (
            processed_dir /
            "open_library_search_clean.csv"
        ).stat().st_size / 1024,
        2
    ),
    "KB"
)

Saved rows: 950
Saved unique works: 950
Saved duplicate work keys: 0
File size: 916.6 KB


### Open Library Search API

The original 1,000 API collection records represented 950 unique Open Library works. Repeated work keys resulted from retrieval overlap across the predefined leadership and management search queries rather than independent book records.

The collection-level records were therefore consolidated into 950 unique work-level records. All original query memberships were preserved in `collection_queries`, while `query_match_count` records the number of predefined search queries through which each work was retrieved.

The consolidation preserved all 1,000 original query memberships while eliminating duplicate work-level records. Query overlap is retained as collection provenance and is not interpreted as a measure of book quality or popularity.

# Notebook 03 — Final Status

**Status: Complete**

Three cleaned and validated source datasets were produced:

1. `open_library_search_clean.csv` — 950 unique Open Library works
2. `open_library_work_enrichment_clean.csv` — 950 corresponding Work API records
3. `leadershipnow_books_clean.csv` — 1,124 cleaned LeadershipNow source records

The cleaning process preserved source provenance, genuine missingness, query retrieval relationships, questionable source identifiers, and archive-level duplicate information rather than removing or imputing them without evidence.

Book-level integration and cross-source deduplication will be performed in Notebook 04.

Open Library Search — 950
            │
            │ openlibrary_key
            ▼
Open Library Work — 950
            │
            ▼
Integrated Open Library — 950
            │
            │ validated ISBN matching
            ▼
LeadershipNow — 1,124 source records
            │
            ▼
Cross-source matching
            │
            ├── Exact ISBN
            ├── Conservative title + author
            └── Unmatched records
            │
            ▼
Canonical Book Dataset